# T2M Diagnostic Benchmark — Clean Pilot Notebook v3

This notebook evaluates pre-generated Text-to-Motion outputs at the atomic-requirement level.  
It uses standardised motion arrays, Human Gold labels, evidence extractors, provisional decision rules, and cross-model validation.

The notebook keeps the workflow explicit and reproducible. **STEP 4 (model selection) must not be skipped.**


## How to Use This Notebook — Multi-model Pilot Evaluation

Use the same benchmark definition and evaluator rules for every model.

Workflow:

1. Prepare/import standardised motion files.
2. Select the model in **STEP 4**.
3. Load the Pilot benchmark definition.
4. Validate all motion files.
5. Construct evaluation cases.
6. Load Human Gold labels.
7. Extract measurable evidence.
8. Calibrate a provisional rule on determinate Human Gold cases.
9. Validate the frozen provisional rule on the current model.
10. Apply the same rule to other models for cross-model validation.
11. Analyse mismatches before changing any rule or threshold.


## Runtime Reset Execution Rule

After a Colab runtime reset, variables and files under `/content` may disappear. Re-run the notebook from the beginning in order.

In particular, do not skip **STEP 4**, because it defines the model and input folder used by later cells.


# Phase A — Benchmark Input Preparation


## STEP 1 — Import Required Libraries

Import the Python libraries used throughout the benchmark.


In [ ]:
import numpy as np

from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Any, Optional

print("STEP 1 ready ✅")



## STEP 2 — Benchmark-wide Configuration

Define shared constants and supported requirement types.

Standardised Motion contract:

- Shape: `[T, 22, 3]`
- Global XYZ coordinates
- +X = Right
- +Y = Up
- +Z = Forward
- Units = metres
- Frame rate = 20 fps


In [ ]:
EXPECTED_JOINTS = 22
EXPECTED_DIMS = 3
DEFAULT_FPS = 20.0

SUPPORTED_REQUIREMENT_TYPES = {
    "action",
    "count",
    "order",
    "direction",
    "turn_direction",
    "attribute_speed",
    "simultaneous",
}

print("STEP 2 ready ✅")



## STEP 3 — Benchmark Input Contract

This notebook evaluates pre-generated `.npy` motion files directly rather than generating motion inside the evaluator.

Every input motion must already satisfy the Standardised Motion contract:

\[
M \in \mathbb{R}^{T\times22\times3}
\]

with global XYZ coordinates, metres, +Y up, +Z forward, +X right, and 20 fps.


In [ ]:
def validate_standardised_motion(motion_xyz):
    """Validate the numerical parts of the benchmark input contract."""
    if not isinstance(motion_xyz, np.ndarray):
        raise TypeError(f"Expected NumPy array, but received {type(motion_xyz)}")
    if motion_xyz.ndim != 3:
        raise ValueError(f"Expected [T, 22, 3], but received {motion_xyz.shape}")
    if motion_xyz.shape[1:] != (EXPECTED_JOINTS, EXPECTED_DIMS):
        raise ValueError(f"Expected [T, 22, 3], but received {motion_xyz.shape}")
    if motion_xyz.shape[0] < 1:
        raise ValueError("Motion contains no frames.")
    if not np.isfinite(motion_xyz).all():
        raise ValueError("Motion contains NaN or Inf values.")
    return motion_xyz

print("STEP 3 — Input Contract ready ✅")



# STEP 3.5 — Import Pre-generated Motion Package

Import the package/folder containing the standardised `.npy` motions to be evaluated.


In [ ]:
# ============================================================
# STEP 3.5 — Import Pre-generated Motion Package
# ============================================================

from google.colab import files
from pathlib import Path
import zipfile

# English note: benchmark logic unchanged from the source notebook.
BENCHMARK_INPUT_ROOT = Path("/content/benchmark_inputs")
BENCHMARK_INPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Upload the standardized motion ZIP:")
print("Example: MoMADiff_standardized_pilot.zip")

# English note: benchmark logic unchanged from the source notebook.
uploaded = files.upload()

zip_files = [
    name for name in uploaded
    if name.lower().endswith(".zip")
]

if len(zip_files) != 1:
    raise RuntimeError(
        f"Expected exactly 1 ZIP file, found {len(zip_files)}."
    )

zip_path = Path(zip_files[0])

# English note: benchmark logic unchanged from the source notebook.
with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(BENCHMARK_INPUT_ROOT)

print("\nPackage extracted ✅")
print("ZIP        :", zip_path.name)
print("Destination:", BENCHMARK_INPUT_ROOT)

# English note: benchmark logic unchanged from the source notebook.
print("\nExtracted files:")
for path in sorted(BENCHMARK_INPUT_ROOT.rglob("*")):
    if path.is_file():
        print(" -", path.relative_to(BENCHMARK_INPUT_ROOT))

print("\nSTEP 3.5 — Motion package ready ✅")



## STEP 4 — Select Evaluation Model and Input Folder

**Do not skip this step.**

Select the T2M model currently being evaluated and set its motion input directory.  
All later evaluation cells use this selection. When switching models for cross-model validation, return to STEP 4 and select the new model without changing the evaluator rules.


In [ ]:
from pathlib import Path

# ============================================================
# STEP 4 — Select Model and Standardised Motion Folder
# ============================================================

# English note: benchmark logic unchanged from the source notebook.
MODEL_NAME = "MoMADiff"

# English note: benchmark logic unchanged from the source notebook.
BENCHMARK_INPUT_ROOT = Path("/content/benchmark_inputs")

# English note: benchmark logic unchanged from the source notebook.
# /content/benchmark_inputs/
#   MoMADiff/
#   MotionHiFlow/
#   Light-T2M/
#   T2M-GPT/
#
# English note: benchmark logic unchanged from the source notebook.
#   C1-01.npy
#   C1-02.npy
#   ...

MODEL_INPUT_DIR = BENCHMARK_INPUT_ROOT / MODEL_NAME

print("=" * 80)
print("MODEL SELECTION")
print("=" * 80)
print("Model       :", MODEL_NAME)
print("Input Folder:", MODEL_INPUT_DIR)

if not MODEL_INPUT_DIR.exists():
    raise FileNotFoundError(
        f"Model input folder not found: {MODEL_INPUT_DIR}\n"
        "Check MODEL_NAME and the uploaded motion package."
    )

print("Model input folder found successfully.")



## STEP 5 — Load Pilot Benchmark Definition

Load the frozen Pilot prompts and their machine-readable atomic requirements.  
The evaluator reads these requirements directly; it does not dynamically re-parse the natural-language prompt.


In [ ]:
# ============================================================
# STEP 5 — Upload & Load Pilot Benchmark Definition
# ============================================================

import json
from google.colab import files


# ------------------------------------------------------------
# English note: benchmark logic unchanged from the source notebook.
# ------------------------------------------------------------

print("Pilot Benchmark JSON。")

uploaded = files.upload()

if not uploaded:
    raise RuntimeError("。")


# ------------------------------------------------------------
# English note: benchmark logic unchanged from the source notebook.
# ------------------------------------------------------------

uploaded_filename = next(iter(uploaded))

print(f"\nUploaded file: {uploaded_filename}")


# ------------------------------------------------------------
# English note: benchmark logic unchanged from the source notebook.
# ------------------------------------------------------------

try:
    benchmark_definition = json.loads(
        uploaded[uploaded_filename].decode("utf-8")
    )
except Exception as e:
    raise ValueError(
        f"JSON:\n{uploaded_filename}\n\n{e}"
    )


# ------------------------------------------------------------
# English note: benchmark logic unchanged from the source notebook.
# ------------------------------------------------------------

if "prompts" not in benchmark_definition:
    raise KeyError(
        "JSON 'prompts' 。"
    )

pilot_prompts = benchmark_definition["prompts"]


# ------------------------------------------------------------
# English note: benchmark logic unchanged from the source notebook.
# ------------------------------------------------------------

print("\n========================================")
print("Pilot Benchmark Definition")
print("========================================")
print("File      :", uploaded_filename)
print("Benchmark :", benchmark_definition.get("benchmark_name", "N/A"))
print("Version   :", benchmark_definition.get("schema_version", "N/A"))
print("Prompts   :", len(pilot_prompts))
print("----------------------------------------")

for prompt in pilot_prompts:
    print(
        f'{prompt["prompt_id"]:7} | '
        f'{prompt["capability"]} | '
        f'{prompt["difficulty"]:6} | '
        f'{prompt["text"]}'
    )

print("========================================")
print("STEP 5 — Pilot Benchmark loaded ✅")



## STEP 6 — Load and Validate Pre-generated `.npy` Motions

Load each motion and verify that it satisfies the Standardised Motion contract before evaluation.


In [ ]:
registered_motions = []
input_results = []

print("========================================")
print("Pre-generated Motion Input")
print("========================================")
print("Model        :", MODEL_NAME)
print("Input folder :", MODEL_INPUT_DIR)
print("Total prompts:", len(pilot_prompts))
print("========================================\n")

for index, prompt_data in enumerate(pilot_prompts, start=1):
    prompt_id = prompt_data["prompt_id"]
    prompt_text = prompt_data["text"]
    capability = prompt_data["capability"]
    difficulty = prompt_data["difficulty"]
    target_frames = prompt_data.get("target_frames_20fps")
    motion_path = MODEL_INPUT_DIR / f"{prompt_id}.npy"

    print("----------------------------------------")
    print(f"[{index}/{len(pilot_prompts)}]")
    print("Prompt ID :", prompt_id)
    print("Prompt    :", prompt_text)
    print("File      :", motion_path)

    try:
        if not motion_path.exists():
            raise FileNotFoundError(f"Input motion not found: {motion_path}")

        motion_xyz = np.load(motion_path, allow_pickle=False)
        validated_motion = validate_standardised_motion(motion_xyz)

        # Target frame length is reported, not forced here.
        frame_match = (
            None if target_frames is None
            else int(validated_motion.shape[0]) == int(target_frames)
        )

        record = {
            "model": MODEL_NAME,
            "prompt_id": prompt_id,
            "prompt": prompt_text,
            "capability": capability,
            "difficulty": difficulty,
            "target_frames": target_frames,
            "motion_path": str(motion_path),
            "shape": list(validated_motion.shape),
            "frame_match": frame_match,
            "validation": "PASS",
        }
        registered_motions.append(record)
        input_results.append(record)

        print("Loaded     :", validated_motion.shape)
        print("Frame match:", frame_match)
        print("Validation : PASS ✅")

    except Exception as e:
        input_results.append({
            "model": MODEL_NAME,
            "prompt_id": prompt_id,
            "motion_path": str(motion_path),
            "validation": "FAIL",
            "error": str(e),
        })
        print("Validation : FAIL ❌")
        print("Error      :", e)

print("\n========================================")
print("Input Validation Summary")
print("========================================")
print("PASS:", len(registered_motions))
print("FAIL:", len(pilot_prompts) - len(registered_motions))
print("========================================")



# Phase B — Automatic Evaluation


## STEP 7 — Evaluation Case Construction

Combine each Pilot prompt, its atomic requirements, and the corresponding standardised motion path into a consistent evaluation-case structure.


In [ ]:
motion_registry = {m["prompt_id"]: m for m in registered_motions}
evaluation_cases = []

for prompt_data in pilot_prompts:
    prompt_id = prompt_data["prompt_id"]
    if prompt_id not in motion_registry:
        print(f"{prompt_id}: Motion not available ⚠️")
        continue

    motion_data = motion_registry[prompt_id]
    evaluation_cases.append({
        "model": motion_data["model"],
        "prompt_id": prompt_id,
        "prompt": prompt_data["text"],
        "capability": prompt_data["capability"],
        "difficulty": prompt_data["difficulty"],
        "motion_path": motion_data["motion_path"],
        "motion_shape": motion_data["shape"],
        "requirements": prompt_data["requirements"],
        "diagnostic_axes": prompt_data.get("diagnostic_axes", []),
    })

print("========================================")
print("Evaluation Case Construction")
print("========================================")
print("Model            :", MODEL_NAME)
print("Benchmark Prompts:", len(pilot_prompts))
print("Evaluation Cases :", len(evaluation_cases))
print("========================================")

if evaluation_cases:
    example = evaluation_cases[0]
    print("Example:", example["prompt_id"], example["motion_path"], example["motion_shape"])



## STEP 8 — Evaluation Configuration

Define shared evaluation settings. These settings must remain model-independent during cross-model comparison.


In [ ]:


# English note: benchmark logic unchanged from the source notebook.
EVALUATION_CONFIG = {
    "action": "ActionEvaluator",
    "direction": "TrajectoryEvaluator",
    "torso_direction": "TorsoGeometryEvaluator",
    "body_side": "BodySideEvaluator",
    "arm_direction": "LimbGeometryEvaluator",
    "leg_direction": "LimbGeometryEvaluator",
    "turn_direction": "RotationEvaluator",
    "count": "CountEvaluator",
    "order": "OrderEvaluator",
    "attribute": "AttributeEvaluator",
    "simultaneous": "SimultaneousEvaluator",
    "relation": "SpatialRelationEvaluator",
    "target": "SpatialRelationEvaluator",
}


# English note: benchmark logic unchanged from the source notebook.
used_requirement_types = sorted({
    requirement["type"]
    for case in evaluation_cases
    for requirement in case["requirements"]
})


# English note: benchmark logic unchanged from the source notebook.
missing_types = [
    requirement_type
    for requirement_type in used_requirement_types
    if requirement_type not in EVALUATION_CONFIG
]


# ------------------------------------------------------------
# English note: benchmark logic unchanged from the source notebook.
# ------------------------------------------------------------

print("========================================")
print("Evaluation Configuration")
print("========================================")

for requirement_type in used_requirement_types:
    evaluator = EVALUATION_CONFIG.get(requirement_type)

    print(
        f"{requirement_type:<18} → {evaluator}"
    )

print("========================================")

if not missing_types:
    print("Requirement TypeEvaluator ✅")
else:
    print("EvaluatorRequirement Type ⚠️")

    for requirement_type in missing_types:
        print(" -", requirement_type)



## STEP 9 — Evaluator Registry / Interface

Define the common evaluator interface and registry used to route atomic requirements to the appropriate evidence extractor or event detector.


In [ ]:

from abc import ABC, abstractmethod


# ------------------------------------------------------------
# 1. Common Evaluator Interface
# ------------------------------------------------------------

class BaseEvaluator(ABC):
    """
    Common interface for all Requirement Evaluators.
    """

    @abstractmethod
    def evaluate(self, motion, requirement, evaluation_case):
        """
        Evaluate one requirement.

        Parameters
        ----------
        motion : np.ndarray
            Standardised motion with shape [T, 22, 3].

        requirement : dict
            Requirement definition from the benchmark JSON.

        evaluation_case : dict
            Prompt-level evaluation information.

        Returns
        -------
        dict
            Requirement-level evaluation result.
        """
        pass


# ------------------------------------------------------------
# 2. Evaluator Registry
# ------------------------------------------------------------

EVALUATOR_REGISTRY = {}


def register_evaluator(name, evaluator_class):
    """
    Register an evaluator class.
    """

    if not issubclass(evaluator_class, BaseEvaluator):
        raise TypeError(
            f"{evaluator_class.__name__} must inherit from BaseEvaluator."
        )

    EVALUATOR_REGISTRY[name] = evaluator_class


def get_evaluator(name):
    """
    Retrieve an evaluator class from the registry.
    """

    if name not in EVALUATOR_REGISTRY:
        raise KeyError(
            f"Evaluator '{name}' is not registered."
        )

    return EVALUATOR_REGISTRY[name]


# ------------------------------------------------------------
# 3. Check Interface / Registry
# ------------------------------------------------------------

print("========================================")
print("Evaluator Registry / Interface")
print("========================================")

print("Base Interface : BaseEvaluator")
print("Registered Evaluators :", len(EVALUATOR_REGISTRY))

print("========================================")
print("Evaluator interface and registry are ready ✅")



# STEP 9C — Human Gold Label Template

Create a Human Gold template at the **requirement level**.

Allowed labels:

- `PASS`
- `FAIL`
- `UNCERTAIN`

`UNCERTAIN` cases are retained for inspection but excluded from threshold fitting and agreement accuracy.


In [ ]:
# ============================================================
# STEP 9C — Human Gold Label Template
# English note: benchmark logic unchanged from the source notebook.
# ============================================================

print("=" * 90)
print("HUMAN GOLD LABEL TEMPLATE")
print("=" * 90)

print(f"\nTotal Evaluation Cases: {len(evaluation_cases)}")


human_gold_template = {}


for case in evaluation_cases:

    # --------------------------------------------------------
    # English note: benchmark logic unchanged from the source notebook.
    # --------------------------------------------------------
    prompt_id = (
        case.get("prompt_id")
        or case.get("id")
        or case.get("case_id")
        or "UNKNOWN"
    )

    # English note: benchmark logic unchanged from the source notebook.
    prompt_text = (
        case.get("prompt")
        or case.get("text")
        or case.get("prompt_text")
        or ""
    )

    requirements = case.get("requirements", [])

    # --------------------------------------------------------
    # English note: benchmark logic unchanged from the source notebook.
    # --------------------------------------------------------
    human_gold_template[prompt_id] = {
        "prompt": prompt_text,
        "requirements": []
    }


    print("\n" + "=" * 90)
    print(f"Prompt ID : {prompt_id}")

    if prompt_text:
        print(f"Prompt    : {prompt_text}")

    print(f"Requirements: {len(requirements)}")
    print("-" * 90)


    # --------------------------------------------------------
    # English note: benchmark logic unchanged from the source notebook.
    # --------------------------------------------------------
    for i, req in enumerate(requirements):

        req_type = req.get("type", "unknown")

        req_value = req.get(
            "value",
            req.get(
                "expected",
                req.get("target")
            )
        )

        # English note: benchmark logic unchanged from the source notebook.
        requirement_entry = {
            "requirement_index": i,
            "type": req_type,
            "value": req_value,
            "human_label": None,
        }

        human_gold_template[
            prompt_id
        ]["requirements"].append(
            requirement_entry
        )


        print(
            f"[{i}] "
            f"{req_type:<15} "
            f"= {str(req_value):<25} "
            f"Human Gold: ?"
        )


print("\n" + "=" * 90)
print("Template created.")
print("Human Gold Label。")
print("=" * 90)



# STEP 9D — Human Gold Label Entry (only when creating new labels)

Enter Human Gold labels for each atomic requirement. Existing saved labels should be loaded instead of overwritten.


In [ ]:
# ============================================================
# STEP 9D — Human Gold Label Entry
# ============================================================

VALID_LABELS = {
    "P": "PASS",
    "F": "FAIL",
    "U": "UNCERTAIN",
}

HUMAN_GOLD_LABELS = {}

print("=" * 90)
print(f"HUMAN GOLD LABEL ENTRY — {MODEL_NAME}")
print("=" * 90)

for prompt_id, data in human_gold_template.items():

    print("\n" + "=" * 90)
    print(f"Prompt ID : {prompt_id}")
    print(f"Prompt    : {data['prompt']}")
    print("-" * 90)

    HUMAN_GOLD_LABELS[prompt_id] = {
        "prompt": data["prompt"],
        "requirements": [],
    }

    for req in data["requirements"]:
        idx = req["requirement_index"]
        typ = req["type"]
        value = req["value"]

        while True:
            x = input(
                f"[{idx}] {typ} = {value} → Human Gold [P/F/U]: "
            ).strip().upper()

            if x in VALID_LABELS:
                break

            print("Invalid input. Please enter P, F, or U.")

        HUMAN_GOLD_LABELS[prompt_id]["requirements"].append({
            "requirement_index": idx,
            "type": typ,
            "value": value,
            "human_label": VALID_LABELS[x],
        })

print("\nHuman Gold Label entry completed.")



# STEP 9E — Save Human Gold Labels as JSON (only when creating new labels)

Save the requirement-level Human Gold labels so the same labels can be reused reproducibly.


In [ ]:
# ============================================================
# STEP 9E — Save Human Gold Labels
# ============================================================

import json
from google.colab import files

gold_filename = f"{MODEL_NAME}_pilot_human_gold_labels.json"

with open(gold_filename, "w", encoding="utf-8") as f:
    json.dump(
        HUMAN_GOLD_LABELS,
        f,
        indent=2,
        ensure_ascii=False,
    )

print("Saved:", gold_filename)

# English note: benchmark logic unchanged from the source notebook.
files.download(gold_filename)



# STEP 9F — Load Saved Human Gold Labels

Load the previously saved Human Gold JSON file.


In [ ]:
# ============================================================
# STEP 9F — Load Saved Human Gold Labels
# ============================================================

import json
from google.colab import files

print(f"Upload Human Gold JSON for: {MODEL_NAME}")
uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No Human Gold JSON was uploaded.")

filename = next(iter(uploaded))

with open(filename, "r", encoding="utf-8") as f:
    HUMAN_GOLD_LABELS = json.load(f)

label_counts = {"PASS": 0, "FAIL": 0, "UNCERTAIN": 0}
total_requirements = 0

for prompt_id, data in HUMAN_GOLD_LABELS.items():
    for req in data["requirements"]:
        label = req.get("human_label")

        if label not in label_counts:
            raise ValueError(
                f"Invalid Human Gold Label: {prompt_id} -> {label}"
            )

        label_counts[label] += 1
        total_requirements += 1

print("\n" + "=" * 80)
print("HUMAN GOLD LABELS LOADED")
print("=" * 80)
print("Model              :", MODEL_NAME)
print("Pilot Prompts      :", len(HUMAN_GOLD_LABELS))
print("Total Requirements :", total_requirements)
print("PASS               :", label_counts["PASS"])
print("FAIL               :", label_counts["FAIL"])
print("UNCERTAIN          :", label_counts["UNCERTAIN"])
print("=" * 80)



## STEP 9G — Human Gold Label Helper

Define a helper that retrieves the Human Gold label for a specific Prompt ID and requirement index while checking that the requirement type/value still matches.


In [ ]:
# ============================================================
# STEP 9G — Human Gold Label Helper
# ============================================================

if "HUMAN_GOLD_LABELS" not in globals():
    raise NameError(
        "HUMAN_GOLD_LABELS is not defined. Run STEP 9F first."
    )

def get_human_label(case, requirement_index):
    """English documentation: this function/class preserves the original benchmark logic. See parameter names, comments, and returned fields."""
    prompt_id = (
        case.get("prompt_id")
        or case.get("id")
        or case.get("case_id")
    )

    if prompt_id is None:
        raise ValueError("Evaluation CasePrompt ID。")

    if prompt_id not in HUMAN_GOLD_LABELS:
        raise KeyError(f"Human Gold Label: {prompt_id}")

    gold_requirements = HUMAN_GOLD_LABELS[prompt_id]["requirements"]

    if requirement_index >= len(gold_requirements):
        raise IndexError(
            f"{prompt_id}: Requirement Index {requirement_index} Human Gold。"
        )

    case_req = case["requirements"][requirement_index]
    gold_req = gold_requirements[requirement_index]

    case_type = case_req.get("type")
    case_value = case_req.get("value", case_req.get("expected"))
    gold_type = gold_req.get("type")
    gold_value = gold_req.get("value")

    if case_type != gold_type or case_value != gold_value:
        raise ValueError(
            f"Requirement mismatch detected.\n"
            f"Prompt ID : {prompt_id}\n"
            f"Index     : {requirement_index}\n"
            f"Case      : {case_type} = {case_value}\n"
            f"Human Gold: {gold_type} = {gold_value}"
        )

    return gold_req["human_label"]

print("✓ HUMAN_GOLD_LABELS loaded:", len(HUMAN_GOLD_LABELS))
print("✓ get_human_label() defined.")



## STEP 10 — Build Requirement Evaluators

For each requirement family, follow the same methodology:

\[
\text{Motion} \rightarrow \text{Evidence} \rightarrow \text{Candidate Rule}
\rightarrow \text{PASS/FAIL} \rightarrow \text{Human Gold Validation}
\rightarrow \text{Cross-model Validation}
\]

A mismatch does **not** automatically mean that a threshold should be changed. Inspect the evidence, segmentation, extra conditions, and evaluator design first.


## STEP 10A — TrajectoryEvaluator: Direction Evidence

Use the root trajectory to measure directional movement.

For root position \(p_t=(x_t,y_t,z_t)\),

\[
\Delta x=x_{T-1}-x_0,\qquad
\Delta z=z_{T-1}-z_0
\]

Required-direction displacement is defined as:

\[
d_{\text{req}}=
\begin{cases}
+\Delta z & \text{forward}\\
-\Delta z & \text{backward}\\
+\Delta x & \text{right}\\
-\Delta x & \text{left}
\end{cases}
\]

Additional evidence includes total horizontal displacement, dominant axis, dominance ratio, and raw direction.  
This step extracts evidence only; the decision rule is defined later.


In [ ]:
# ============================================================
# STEP 10A — TrajectoryEvaluator
# English note: benchmark logic unchanged from the source notebook.
# ============================================================

import numpy as np


class TrajectoryEvaluator:
    """English documentation: this function/class preserves the original benchmark logic. See parameter names, comments, and returned fields."""

    # English note: benchmark logic unchanged from the source notebook.
    SUPPORTED_DIRECTIONS = {
        "forward",
        "backward",
        "left",
        "right",
    }

    def __init__(self):
        pass

    # --------------------------------------------------------
    # English note: benchmark logic unchanged from the source notebook.
    # --------------------------------------------------------
    def extract_trajectory(self, motion):

        motion = np.asarray(motion)

        # English note: benchmark logic unchanged from the source notebook.
        if motion.ndim != 3:
            raise ValueError(
                f"MotionShape [T, J, 3] 。"
                f"Shape: {motion.shape}"
            )

        # English note: benchmark logic unchanged from the source notebook.
        if motion.shape[2] != 3:
            raise ValueError(
                f"XYZ。Shape: {motion.shape}"
            )

        # English note: benchmark logic unchanged from the source notebook.
        root_xyz = motion[:, 0, :]

        # English note: benchmark logic unchanged from the source notebook.
        # English note: benchmark logic unchanged from the source notebook.
        trajectory_xz = root_xyz[:, [0, 2]]

        return trajectory_xz

    # --------------------------------------------------------
    # English note: benchmark logic unchanged from the source notebook.
    # --------------------------------------------------------
    def calculate_evidence(self, motion):

        # English note: benchmark logic unchanged from the source notebook.
        trajectory = self.extract_trajectory(motion)

        # English note: benchmark logic unchanged from the source notebook.
        start = trajectory[0]
        end = trajectory[-1]

        # English note: benchmark logic unchanged from the source notebook.
        displacement = end - start

        # English note: benchmark logic unchanged from the source notebook.
        dx = float(displacement[0])

        # English note: benchmark logic unchanged from the source notebook.
        dz = float(displacement[1])

        # English note: benchmark logic unchanged from the source notebook.
        total_displacement = float(
            np.linalg.norm(displacement)
        )

        # English note: benchmark logic unchanged from the source notebook.
        abs_dx = abs(dx)
        abs_dz = abs(dz)

        # ----------------------------------------------------
        # English note: benchmark logic unchanged from the source notebook.
        # ----------------------------------------------------
        if abs_dx > abs_dz:
            dominant_axis = "X"

        elif abs_dz > abs_dx:
            dominant_axis = "Z"

        else:
            dominant_axis = "equal"

        # ----------------------------------------------------
        # English note: benchmark logic unchanged from the source notebook.
        #
        # English note: benchmark logic unchanged from the source notebook.
        # English note: benchmark logic unchanged from the source notebook.
        # English note: benchmark logic unchanged from the source notebook.
        #
        # dominance_ratio = 2.0 / 0.5 = 4.0
        #
        # English note: benchmark logic unchanged from the source notebook.
        # ----------------------------------------------------
        minor = min(abs_dx, abs_dz)
        major = max(abs_dx, abs_dz)

        if minor > 1e-8:
            dominance_ratio = major / minor

        elif major > 1e-8:
            dominance_ratio = float("inf")

        else:
            # English note: benchmark logic unchanged from the source notebook.
            dominance_ratio = 0.0

        # ----------------------------------------------------
        # English note: benchmark logic unchanged from the source notebook.
        #
        # English note: benchmark logic unchanged from the source notebook.
        # ----------------------------------------------------
        if dominant_axis == "X":

            # +X = Right
            # -X = Left
            raw_direction = "right" if dx > 0 else "left"

        elif dominant_axis == "Z":

            # +Z = Forward
            # -Z = Backward
            raw_direction = "forward" if dz > 0 else "backward"

        else:
            raw_direction = "undetermined"

        # English note: benchmark logic unchanged from the source notebook.
        return {
            "trajectory_xz": trajectory,

            "start_x": float(start[0]),
            "start_z": float(start[1]),

            "end_x": float(end[0]),
            "end_z": float(end[1]),

            "dx": dx,
            "dz": dz,

            "total_displacement": total_displacement,

            "abs_dx": abs_dx,
            "abs_dz": abs_dz,

            "dominant_axis": dominant_axis,
            "dominance_ratio": dominance_ratio,

            "raw_direction": raw_direction,
        }

    # --------------------------------------------------------
    # English note: benchmark logic unchanged from the source notebook.
    # --------------------------------------------------------
    def evaluate(self, motion, required_direction):

        # English note: benchmark logic unchanged from the source notebook.
        # English note: benchmark logic unchanged from the source notebook.
        required_direction = required_direction.lower()

        # English note: benchmark logic unchanged from the source notebook.
        if required_direction not in self.SUPPORTED_DIRECTIONS:
            raise ValueError(
                f"Direction: {required_direction}"
            )

        # English note: benchmark logic unchanged from the source notebook.
        evidence = self.calculate_evidence(motion)

        # ----------------------------------------------------
        # English note: benchmark logic unchanged from the source notebook.
        #
        # English note: benchmark logic unchanged from the source notebook.
        # English note: benchmark logic unchanged from the source notebook.
        #
        # English note: benchmark logic unchanged from the source notebook.
        #
        # English note: benchmark logic unchanged from the source notebook.
        # English note: benchmark logic unchanged from the source notebook.
        #
        # English note: benchmark logic unchanged from the source notebook.
        # ----------------------------------------------------

        return {
            # English note: benchmark logic unchanged from the source notebook.
            "requirement_type": "direction",

            # English note: benchmark logic unchanged from the source notebook.
            "required_direction": required_direction,

            # English note: benchmark logic unchanged from the source notebook.
            "predicted_direction_raw":
                evidence["raw_direction"],

            # English note: benchmark logic unchanged from the source notebook.
            "pass_fail": None,

            # English note: benchmark logic unchanged from the source notebook.
            "evidence": evidence,
        }


print("TrajectoryEvaluator。")



## STEP 10A-1 — Quick Sanity Check

Before calibration, run one registered Direction case and verify that the motion loads correctly and that the trajectory evidence has the expected sign and scale.


In [ ]:
# English note: benchmark logic unchanged from the source notebook.
left_case = None
for case in evaluation_cases:
    for req in case["requirements"]:
        value = req.get("value", req.get("expected"))
        if req.get("type") == "direction" and str(value).lower() == "left":
            left_case = case
            break
    if left_case is not None:
        break

if left_case is None:
    raise RuntimeError("direction=left Evaluation Case。")

motion = np.load(left_case["motion_path"], allow_pickle=False)
motion = validate_standardised_motion(motion)

evaluator = TrajectoryEvaluator()
result = evaluator.evaluate(motion=motion, required_direction="left")
evidence = result["evidence"]

print("Motion :", left_case["motion_path"])
print("Shape      :", motion.shape)
print("Direction:", result["required_direction"])
print("Direction:", result["predicted_direction_raw"])
print(f"dx: {evidence['dx']:.4f} m")
print(f"dz: {evidence['dz']:.4f} m")
print("Dominant axis :", evidence["dominant_axis"])
print("Dominance ratio:", evidence["dominance_ratio"])
print("PASS / FAIL   :", result["pass_fail"], "(Threshold)")



## STEP 10B — Collect Direction Evidence and Link Human Gold

For every `direction` requirement, collect trajectory evidence and attach the corresponding Human Gold label.

This creates the calibration table:

`Human Gold ↔ measurable Direction evidence`


In [ ]:
# ============================================================
# STEP 10B — Collect Direction Evidence + Human Gold
# English note: benchmark logic unchanged from the source notebook.
# ============================================================

import numpy as np

# ============================================================
# English note: benchmark logic unchanged from the source notebook.
# ============================================================

def calculate_required_direction_ratio(
    dx,
    dz,
    required_direction,
    eps=1e-8,
):
    """English documentation: this function/class preserves the original benchmark logic. See parameter names, comments, and returned fields."""

    required_direction = required_direction.lower()


    # --------------------------------------------------------
    # English note: benchmark logic unchanged from the source notebook.
    # --------------------------------------------------------

    if required_direction == "left":

        # -X = Left
        required_displacement = -dx

        # English note: benchmark logic unchanged from the source notebook.
        orthogonal_displacement = abs(dz)


    elif required_direction == "right":

        # +X = Right
        required_displacement = dx

        # English note: benchmark logic unchanged from the source notebook.
        orthogonal_displacement = abs(dz)


    elif required_direction == "forward":

        # +Z = Forward
        required_displacement = dz

        # English note: benchmark logic unchanged from the source notebook.
        orthogonal_displacement = abs(dx)


    elif required_direction == "backward":

        # -Z = Backward
        required_displacement = -dz

        # English note: benchmark logic unchanged from the source notebook.
        orthogonal_displacement = abs(dx)


    else:

        raise ValueError(
            f"Unsupported direction: {required_direction}"
        )


    # --------------------------------------------------------
    # English note: benchmark logic unchanged from the source notebook.
    # --------------------------------------------------------

    if orthogonal_displacement > eps:

        ratio = (
            required_displacement
            / orthogonal_displacement
        )

    elif required_displacement > eps:

        ratio = float("inf")

    elif required_displacement < -eps:

        ratio = float("-inf")

    else:

        ratio = 0.0


    return (
        float(required_displacement),
        float(orthogonal_displacement),
        float(ratio),
    )


# ============================================================
# English note: benchmark logic unchanged from the source notebook.
# ============================================================

direction_results = []

evaluator = TrajectoryEvaluator()


for case in evaluation_cases:

    # --------------------------------------------------------
    # English note: benchmark logic unchanged from the source notebook.
    # --------------------------------------------------------

    for req_index, req in enumerate(
        case["requirements"]
    ):

        # English note: benchmark logic unchanged from the source notebook.
        if req.get("type") != "direction":
            continue


        # ----------------------------------------------------
        # Required Direction
        # ----------------------------------------------------

        required_direction = req.get(
            "value",
            req.get("expected")
        )

        if required_direction is None:
            continue

        required_direction = str(
            required_direction
        ).lower()


        # ----------------------------------------------------
        # Prompt ID
        # ----------------------------------------------------

        prompt_id = (
            case.get("prompt_id")
            or case.get("id")
            or case.get("case_id")
            or "UNKNOWN"
        )


        # ----------------------------------------------------
        # English note: benchmark logic unchanged from the source notebook.
        # ----------------------------------------------------

        motion = np.load(
            case["motion_path"],
            allow_pickle=False
        )

        motion = validate_standardised_motion(
            motion
        )


        # ----------------------------------------------------
        # TrajectoryEvaluator
        # ----------------------------------------------------

        result = evaluator.evaluate(
            motion=motion,
            required_direction=required_direction
        )

        evidence = result["evidence"]

        dx = evidence["dx"]
        dz = evidence["dz"]


        # ----------------------------------------------------
        # Required-direction Evidence
        # ----------------------------------------------------

        (
            required_displacement,
            orthogonal_displacement,
            required_ratio,
        ) = calculate_required_direction_ratio(

            dx=dx,

            dz=dz,

            required_direction=required_direction,
        )


        # ----------------------------------------------------
        # Human Gold Label
        #
        # English note: benchmark logic unchanged from the source notebook.
        # ----------------------------------------------------

        human_label = get_human_label(
            case=case,
            requirement_index=req_index,
        )


        # ----------------------------------------------------
        # English note: benchmark logic unchanged from the source notebook.
        # ----------------------------------------------------

        direction_results.append({

            "prompt_id":
                prompt_id,

            "requirement_index":
                req_index,

            "motion_path":
                case["motion_path"],

            "required_direction":
                required_direction,

            "human_label":
                human_label,

            "predicted_raw":
                result["predicted_direction_raw"],

            "dx":
                dx,

            "dz":
                dz,

            "required_displacement":
                required_displacement,

            "orthogonal_displacement":
                orthogonal_displacement,

            "required_ratio":
                required_ratio,

            "dominant_axis":
                evidence["dominant_axis"],

            "dominance_ratio":
                evidence["dominance_ratio"],
        })


# ============================================================
# English note: benchmark logic unchanged from the source notebook.
# ============================================================

print(
    f"Direction Requirement Cases: "
    f"{len(direction_results)}"
)

print("=" * 100)


for i, r in enumerate(
    direction_results,
    start=1
):

    print(f"\n[{i}]")

    print(
        "Prompt ID     :",
        r["prompt_id"]
    )

    print(
        "Req Index     :",
        r["requirement_index"]
    )

    print(
        "Motion        :",
        r["motion_path"]
    )

    print(
        "Required      :",
        r["required_direction"]
    )

    print(
        "Human Label   :",
        r["human_label"]
    )

    print(
        "Predicted Raw :",
        r["predicted_raw"]
    )

    print(
        f"dx            : "
        f"{r['dx']:.4f} m"
    )

    print(
        f"dz            : "
        f"{r['dz']:.4f} m"
    )

    print(
        f"Required Disp : "
        f"{r['required_displacement']:.4f} m"
    )

    print(
        f"Orthogonal    : "
        f"{r['orthogonal_displacement']:.4f} m"
    )

    print(
        f"Required Ratio: "
        f"{r['required_ratio']:.4f}"
    )

    print("-" * 100)


# ============================================================
# English note: benchmark logic unchanged from the source notebook.
# ============================================================

print("\n" + "=" * 100)
print("HUMAN LABEL SUMMARY")
print("=" * 100)

pass_count = sum(
    r["human_label"] == "PASS"
    for r in direction_results
)

fail_count = sum(
    r["human_label"] == "FAIL"
    for r in direction_results
)

uncertain_count = sum(
    r["human_label"] == "UNCERTAIN"
    for r in direction_results
)

none_count = sum(
    r["human_label"] is None
    for r in direction_results
)


print("PASS      :", pass_count)
print("FAIL      :", fail_count)
print("UNCERTAIN :", uncertain_count)
print("None      :", none_count)


if none_count == 0:
    print("\nHuman Gold Labels linked successfully ✅")
else:
    print(
        "\nWARNING: Human Label"
        "Requirement。"
    )


print("=" * 100)
print("STEP 10B — Direction Evidence collected ✅")
print("=" * 100)



## STEP 10C — Search Candidate Minimum Required-Displacement Thresholds

Use only determinate Human Gold cases (`PASS` and `FAIL`) for calibration; exclude `UNCERTAIN`.

For each required direction, compute \(d_{\text{req}}\) from STEP 10A and test candidate thresholds \(\tau_d\).

Candidate rule:

\[
PASS \iff d_{\text{req}} \ge \tau_d
\]

The threshold selected here is provisional and must later be validated across models.


In [ ]:
# ============================================================
# STEP 10C — Candidate Direction Threshold Search
# English note: benchmark logic unchanged from the source notebook.
# ============================================================

import numpy as np


# ------------------------------------------------------------
# English note: benchmark logic unchanged from the source notebook.
# ------------------------------------------------------------

candidate_thresholds = [
    0.10,
    0.25,
    0.50,
    0.75,
    1.00,
    1.25,
]


# ------------------------------------------------------------
# English note: benchmark logic unchanged from the source notebook.
# ------------------------------------------------------------

calibration_cases = [
    r for r in direction_results
    if r["human_label"] in {"PASS", "FAIL"}
]


print("=" * 100)
print("DIRECTION THRESHOLD CALIBRATION")
print("=" * 100)

print(
    "Usable Human-labelled cases:",
    len(calibration_cases)
)

print(
    "UNCERTAIN excluded:",
    len(direction_results) - len(calibration_cases)
)


# ------------------------------------------------------------
# English note: benchmark logic unchanged from the source notebook.
# ------------------------------------------------------------

threshold_results = []


for threshold in candidate_thresholds:

    correct = 0
    total = 0

    false_pass = 0
    false_fail = 0

    case_results = []


    for r in calibration_cases:

        required_disp = r[
            "required_displacement"
        ]

        human_label = r[
            "human_label"
        ]


        # ----------------------------------------------------
        # Candidate Direction Rule
        #
        # English note: benchmark logic unchanged from the source notebook.
        # 2. Required displacement >= threshold
        # ----------------------------------------------------

        if required_disp >= threshold:

            predicted_label = "PASS"

        else:

            predicted_label = "FAIL"


        # ----------------------------------------------------
        # English note: benchmark logic unchanged from the source notebook.
        # ----------------------------------------------------

        is_correct = (
            predicted_label == human_label
        )

        if is_correct:
            correct += 1

        elif (
            predicted_label == "PASS"
            and human_label == "FAIL"
        ):
            false_pass += 1

        elif (
            predicted_label == "FAIL"
            and human_label == "PASS"
        ):
            false_fail += 1


        total += 1


        case_results.append({

            "prompt_id":
                r["prompt_id"],

            "human_label":
                human_label,

            "predicted_label":
                predicted_label,

            "required_displacement":
                required_disp,

            "correct":
                is_correct,
        })


    # --------------------------------------------------------
    # Accuracy
    # --------------------------------------------------------

    accuracy = (
        correct / total
        if total > 0
        else 0.0
    )


    threshold_results.append({

        "threshold":
            threshold,

        "correct":
            correct,

        "total":
            total,

        "accuracy":
            accuracy,

        "false_pass":
            false_pass,

        "false_fail":
            false_fail,

        "cases":
            case_results,
    })


# ------------------------------------------------------------
# English note: benchmark logic unchanged from the source notebook.
# ------------------------------------------------------------

print("\n")
print(
    f"{'Threshold':<12}"
    f"{'Correct':<12}"
    f"{'Accuracy':<12}"
    f"{'False PASS':<14}"
    f"{'False FAIL':<14}"
)

print("-" * 70)


for result in threshold_results:

    print(
        f"{result['threshold']:<12.2f}"
        f"{result['correct']:<12}"
        f"{result['accuracy']:<12.3f}"
        f"{result['false_pass']:<14}"
        f"{result['false_fail']:<14}"
    )


# ------------------------------------------------------------
# English note: benchmark logic unchanged from the source notebook.
# ------------------------------------------------------------

best_accuracy = max(
    r["accuracy"]
    for r in threshold_results
)

best_candidates = [
    r for r in threshold_results
    if r["accuracy"] == best_accuracy
]


print("\n" + "=" * 100)
print("BEST CANDIDATE(S)")
print("=" * 100)


for result in best_candidates:

    print(
        f"Threshold = "
        f"{result['threshold']:.2f} m"
        f" | Accuracy = "
        f"{result['accuracy']:.3f}"
    )


print("=" * 100)
print(
    "NOTE: These are PILOT candidate thresholds, "
    "not final benchmark thresholds."
)
print("=" * 100)



## STEP 10D — Initial Direction Decision Rule

The provisional Direction rule is:

\[
PASS \iff d_{\text{req}} \ge 0.50\text{ m}
\]

where \(d_{\text{req}}\) is the signed displacement toward the required direction.

The `0.50 m` value is a Pilot candidate, not a universal physical definition of successful directional motion.


In [ ]:
# ============================================================
# STEP 10D — Initial Direction Decision Rule
# ============================================================

class DirectionDecisionRule:

    def __init__(self, min_displacement=0.50):
        self.min_displacement = float(min_displacement)

    def evaluate(self, evidence, required_direction):

        required_direction = required_direction.lower()

        dx = float(evidence["dx"])
        dz = float(evidence["dz"])

        if required_direction == "forward":
            required_displacement = dz
            correct_sign = dz > 0

        elif required_direction == "backward":
            required_displacement = -dz
            correct_sign = dz < 0

        elif required_direction == "right":
            required_displacement = dx
            correct_sign = dx > 0

        elif required_direction == "left":
            required_displacement = -dx
            correct_sign = dx < 0

        else:
            raise ValueError(
                f"Unsupported direction: {required_direction}"
            )

        sufficient_displacement = (
            required_displacement >= self.min_displacement
        )

        prediction = (
            "PASS"
            if correct_sign and sufficient_displacement
            else "FAIL"
        )

        return {
            "required_direction": required_direction,
            "prediction": prediction,
            "correct_sign": correct_sign,
            "required_displacement": required_displacement,
            "min_displacement_threshold": self.min_displacement,
            "sufficient_displacement": sufficient_displacement,
            "dominant_axis": evidence.get("dominant_axis"),
            "dominance_ratio": evidence.get("dominance_ratio"),
        }

print("DirectionDecisionRule defined successfully.")



## STEP 10E — Validate the Initial Direction Rule on the Current Model

Apply the frozen provisional Direction rule to the current STEP 4 model and compare automatic decisions with Human Gold.

Report determinate agreement and inspect every mismatch. `UNCERTAIN` cases are displayed but excluded from accuracy.


In [ ]:
# ============================================================
# STEP 10E — Current Model Pilot Validation
# ============================================================

direction_rule = DirectionDecisionRule(
    min_displacement=0.50
)

validation_results = []

print("=" * 110)
print(f"INITIAL DIRECTION RULE — PILOT VALIDATION [{MODEL_NAME}]")
print("=" * 110)

for result in direction_results:

    evidence = {
        "dx": result["dx"],
        "dz": result["dz"],
        "dominant_axis": result["dominant_axis"],
        "dominance_ratio": result["dominance_ratio"],
    }

    auto_result = direction_rule.evaluate(
        evidence=evidence,
        required_direction=result["required_direction"],
    )

    human_label = result["human_label"]
    prediction = auto_result["prediction"]

    if human_label == "UNCERTAIN":
        match = None
        status = "EXCLUDED"
    else:
        match = prediction == human_label
        status = "MATCH" if match else "MISMATCH"

    validation_results.append({
        "model": MODEL_NAME,
        "prompt_id": result["prompt_id"],
        "requirement_index": result["requirement_index"],
        "required_direction": result["required_direction"],
        "human_label": human_label,
        "prediction": prediction,
        "dx": result["dx"],
        "dz": result["dz"],
        "required_displacement": auto_result["required_displacement"],
        "correct_sign": auto_result["correct_sign"],
        "sufficient_displacement": auto_result["sufficient_displacement"],
        "match": match,
        "status": status,
    })

usable = [r for r in validation_results if r["match"] is not None]
correct = sum(1 for r in usable if r["match"])
accuracy = correct / len(usable) if usable else 0.0

false_pass = sum(
    1 for r in usable
    if r["human_label"] == "FAIL" and r["prediction"] == "PASS"
)

false_fail = sum(
    1 for r in usable
    if r["human_label"] == "PASS" and r["prediction"] == "FAIL"
)

print(f"Usable Cases       : {len(usable)}")
print(f"Correct Predictions: {correct}")
print(f"Accuracy           : {accuracy:.3f}")
print(f"False PASS         : {false_pass}")
print(f"False FAIL         : {false_fail}")
print(f"UNCERTAIN Excluded : {len(validation_results) - len(usable)}")

mismatches = [r for r in usable if not r["match"]]

if mismatches:
    print("\nMISMATCH CASES")
    print("-" * 110)
    for r in mismatches:
        print(
            r["prompt_id"],
            "| Required:", r["required_direction"],
            "| Human:", r["human_label"],
            "| Auto:", r["prediction"],
            "| Required displacement:",
            f"{r['required_displacement']:.4f} m",
        )
else:
    print("\nNo PASS/FAIL mismatches found for this model.")



## STEP 10F — Cross-model Direction Validation (run on other models)

Keep the Direction rule unchanged, switch models in STEP 4, and apply the same rule.  
Only revise the rule after analysing cross-model mismatches.


## STEP 10G — RotationEvaluator: Turn Direction Evidence

Estimate body yaw from the hip orientation.

Let the lateral hip vector be:

\[
l_t = R_t-L_t
\]

Define the horizontal body heading:

\[
h_t=(-l_{t,z},\,l_{t,x})
\]

Yaw is:

\[
\theta_t=\operatorname{atan2}(h_{t,x},h_{t,z})
\]

Unwrap the angle and normalise it relative to the first frame:

\[
\tilde{\theta}_t=\operatorname{unwrap}(\theta_t)-\theta_0
\]

Net rotation:

\[
R_{\text{net}}=\tilde{\theta}_{T-1}-\tilde{\theta}_0
\]

Total absolute rotation:

\[
R_{\text{total}}=\sum_{t=1}^{T-1}
|\tilde{\theta}_t-\tilde{\theta}_{t-1}|
\]

In this benchmark, positive rotation is Right and negative rotation is Left.  
This step extracts evidence only.


In [ ]:
# ============================================================
# STEP 10G — RotationEvaluator
# English note: benchmark logic unchanged from the source notebook.
# English note: benchmark logic unchanged from the source notebook.
# ============================================================

import numpy as np


class RotationEvaluator:
    """English documentation: this function/class preserves the original benchmark logic. See parameter names, comments, and returned fields."""

    SUPPORTED_TURN_DIRECTIONS = {
        "left",
        "right",
    }


    # --------------------------------------------------------
    # English note: benchmark logic unchanged from the source notebook.
    # --------------------------------------------------------

    def _validate_motion(self, motion):

        motion = np.asarray(motion)

        if motion.ndim != 3:
            raise ValueError(
                f"motion must have shape [T, J, 3], "
                f"but got {motion.shape}"
            )

        if motion.shape[2] != 3:
            raise ValueError(
                f"Last dimension must be XYZ (=3), "
                f"but got {motion.shape}"
            )

        if motion.shape[1] < 3:
            raise ValueError(
                "At least joints 0, 1, 2 are required."
            )

        return motion


    # --------------------------------------------------------
    # English note: benchmark logic unchanged from the source notebook.
    # --------------------------------------------------------

    def _compute_heading_vectors(self, motion):
        """English documentation: this function/class preserves the original benchmark logic. See parameter names, comments, and returned fields."""

        left_hip = motion[:, 1, :]
        right_hip = motion[:, 2, :]

        # Left Hip → Right Hip
        lateral = right_hip - left_hip

        # English note: benchmark logic unchanged from the source notebook.
        lateral_xz = lateral[:, [0, 2]]

        # English note: benchmark logic unchanged from the source notebook.
        norms = np.linalg.norm(
            lateral_xz,
            axis=1,
            keepdims=True
        )

        norms = np.maximum(
            norms,
            1e-8
        )

        lateral_xz = (
            lateral_xz / norms
        )

        # English note: benchmark logic unchanged from the source notebook.
        heading = np.stack(
            [
                -lateral_xz[:, 1],
                lateral_xz[:, 0]
            ],
            axis=1
        )

        return heading


    # --------------------------------------------------------
    # Heading → Yaw Angle
    # --------------------------------------------------------

    def _heading_to_yaw(self, heading):
        """English documentation: this function/class preserves the original benchmark logic. See parameter names, comments, and returned fields."""

        x = heading[:, 0]
        z = heading[:, 1]

        yaw_rad = np.arctan2(
            x,
            z
        )

        # English note: benchmark logic unchanged from the source notebook.
        yaw_rad = np.unwrap(
            yaw_rad
        )

        return np.degrees(
            yaw_rad
        )


    # --------------------------------------------------------
    # English note: benchmark logic unchanged from the source notebook.
    # --------------------------------------------------------

    def extract_evidence(self, motion):

        motion = self._validate_motion(
            motion
        )

        heading = (
            self._compute_heading_vectors(
                motion
            )
        )

        yaw_deg = (
            self._heading_to_yaw(
                heading
            )
        )

        initial_yaw = float(
            yaw_deg[0]
        )

        final_yaw = float(
            yaw_deg[-1]
        )

        # English note: benchmark logic unchanged from the source notebook.
        net_rotation = float(
            final_yaw
            - initial_yaw
        )

        # English note: benchmark logic unchanged from the source notebook.
        if len(yaw_deg) > 1:

            total_rotation = float(
                np.sum(
                    np.abs(
                        np.diff(yaw_deg)
                    )
                )
            )

        else:

            total_rotation = 0.0


        # ----------------------------------------------------
        # Raw Turn Direction
        #
        # English note: benchmark logic unchanged from the source notebook.
        # positive yaw = right
        # negative yaw = left
        #
        # English note: benchmark logic unchanged from the source notebook.
        # ----------------------------------------------------

        if net_rotation > 0:

            raw_turn_direction = "right"

        elif net_rotation < 0:

            raw_turn_direction = "left"

        else:

            raw_turn_direction = "none"


        return {

            "initial_yaw_deg":
                initial_yaw,

            "final_yaw_deg":
                final_yaw,

            "net_rotation_deg":
                net_rotation,

            "total_rotation_deg":
                total_rotation,

            "raw_turn_direction":
                raw_turn_direction,

            "yaw_trajectory_deg":
                yaw_deg,

            "heading_vectors":
                heading,

            # English note: benchmark logic unchanged from the source notebook.
            "pass_fail":
                None,
        }


print(
    "RotationEvaluator defined successfully."
)



### STEP 10G-1 — Quick Sanity Check

Use one registered `turn_direction = right` motion to verify:

- motion loading,
- hip-based body heading,
- yaw calculation,
- initial/final yaw,
- net and total rotation,
- raw turn direction.

No PASS/FAIL calibration is performed here.


In [ ]:
# ============================================================
# STEP 10G-1 — Quick Sanity Check
#
# English note: benchmark logic unchanged from the source notebook.
# English note: benchmark logic unchanged from the source notebook.
#
# English note: benchmark logic unchanged from the source notebook.
# ============================================================


# ------------------------------------------------------------
# English note: benchmark logic unchanged from the source notebook.
# English note: benchmark logic unchanged from the source notebook.
# ------------------------------------------------------------

right_turn_case = None

for case in evaluation_cases:

    for req in case["requirements"]:

        value = req.get(
            "value",
            req.get("expected")
        )

        if (
            req.get("type") == "turn_direction"
            and str(value).lower() == "right"
        ):
            right_turn_case = case
            break

    if right_turn_case is not None:
        break


# ------------------------------------------------------------
# English note: benchmark logic unchanged from the source notebook.
# ------------------------------------------------------------

if right_turn_case is None:

    raise RuntimeError(
        "turn_direction=right "
        "Evaluation Case。"
    )


# ------------------------------------------------------------
# English note: benchmark logic unchanged from the source notebook.
# ------------------------------------------------------------

motion = np.load(
    right_turn_case["motion_path"],
    allow_pickle=False
)

motion = validate_standardised_motion(
    motion
)


# ------------------------------------------------------------
# English note: benchmark logic unchanged from the source notebook.
# ------------------------------------------------------------

evaluator = RotationEvaluator()

evidence = evaluator.extract_evidence(
    motion=motion
)


# ------------------------------------------------------------
# English note: benchmark logic unchanged from the source notebook.
# ------------------------------------------------------------

print("========================================")
print("RotationEvaluator — Sanity Check")
print("========================================")

print(
    "Motion      :",
    right_turn_case["motion_path"]
)

print(
    "Shape           :",
    motion.shape
)

print(
    "Turn        :",
    "right"
)

print(
    "Turn        :",
    evidence["raw_turn_direction"]
)

print("----------------------------------------")

print(
    f"Initial yaw     : "
    f"{evidence['initial_yaw_deg']:.2f}°"
)

print(
    f"Final yaw       : "
    f"{evidence['final_yaw_deg']:.2f}°"
)

print(
    f"Net rotation    : "
    f"{evidence['net_rotation_deg']:.2f}°"
)

print(
    f"Total rotation  : "
    f"{evidence['total_rotation_deg']:.2f}°"
)

print(
    "PASS / FAIL     :",
    evidence["pass_fail"]
)


# ------------------------------------------------------------
# English note: benchmark logic unchanged from the source notebook.
# ------------------------------------------------------------

print("----------------------------------------")
print("Sanity Check")

if evidence["raw_turn_direction"] == "right":

    print(
        "✓ Raw Turn Direction"
        "right。"
    )

else:

    print(
        "⚠ Raw Turn Direction"
        "right。"
    )

    print(
        "Yaw Angle"
        "Body Heading。"
    )


# ------------------------------------------------------------
# English note: benchmark logic unchanged from the source notebook.
# English note: benchmark logic unchanged from the source notebook.
# ------------------------------------------------------------

if evidence["pass_fail"] is None:

    print(
        "✓ PASS / FAIL"
        "（Calibration）。"
    )

else:

    print(
        "⚠ Calibration"
        "PASS / FAILNone。"
    )



## STEP 10H — Collect Turn Direction Evidence and Link Human Gold

For every `turn_direction` requirement, collect Rotation evidence and attach the corresponding Human Gold label.

Recorded evidence includes Prompt ID, requirement index, required turn direction, initial/final yaw, net rotation, total rotation, and raw turn direction.


In [ ]:
# ============================================================
# STEP 10H — Collect Turn Direction Evidence + Human Gold
#
# English note: benchmark logic unchanged from the source notebook.
# English note: benchmark logic unchanged from the source notebook.
# ============================================================

import numpy as np
import pandas as pd


rotation_evaluator = RotationEvaluator()

turn_evidence_records = []


# ------------------------------------------------------------
# English note: benchmark logic unchanged from the source notebook.
# ------------------------------------------------------------

for case in evaluation_cases:

    # English note: benchmark logic unchanged from the source notebook.
    motion = np.load(
        case["motion_path"],
        allow_pickle=False
    )

    motion = validate_standardised_motion(
        motion
    )


    # --------------------------------------------------------
    # English note: benchmark logic unchanged from the source notebook.
    # --------------------------------------------------------

    for requirement_index, req in enumerate(
        case["requirements"]
    ):

        # English note: benchmark logic unchanged from the source notebook.
        if req.get("type") != "turn_direction":
            continue


        # ----------------------------------------------------
        # English note: benchmark logic unchanged from the source notebook.
        # ----------------------------------------------------

        required_turn = req.get(
            "value",
            req.get("expected")
        )

        required_turn = str(
            required_turn
        ).lower()


        # ----------------------------------------------------
        # English note: benchmark logic unchanged from the source notebook.
        #
        # English note: benchmark logic unchanged from the source notebook.
        # ----------------------------------------------------

        human_label = get_human_label(
            case,
            requirement_index
        )


        # ----------------------------------------------------
        # English note: benchmark logic unchanged from the source notebook.
        # ----------------------------------------------------

        evidence = (
            rotation_evaluator.extract_evidence(
                motion=motion
            )
        )


        # ----------------------------------------------------
        # English note: benchmark logic unchanged from the source notebook.
        # ----------------------------------------------------

        turn_evidence_records.append({

            "prompt_id":
                case.get("prompt_id"),

            "requirement_index":
                requirement_index,

            "required_turn":
                required_turn,

            "human_label":
                human_label,

            "initial_yaw_deg":
                evidence["initial_yaw_deg"],

            "final_yaw_deg":
                evidence["final_yaw_deg"],

            "net_rotation_deg":
                evidence["net_rotation_deg"],

            "total_rotation_deg":
                evidence["total_rotation_deg"],

            "raw_turn_direction":
                evidence["raw_turn_direction"],

            "motion_path":
                case["motion_path"],
        })


# ------------------------------------------------------------
# English note: benchmark logic unchanged from the source notebook.
# ------------------------------------------------------------

turn_evidence_df = pd.DataFrame(
    turn_evidence_records
)


# ------------------------------------------------------------
# English note: benchmark logic unchanged from the source notebook.
# ------------------------------------------------------------

print("=" * 100)
print(
    "STEP 10H — Turn Direction Evidence + Human Gold"
)
print("=" * 100)

print(
    "Total Turn Direction Requirements:",
    len(turn_evidence_df)
)

print("-" * 100)


if len(turn_evidence_df) == 0:

    print(
        "turn_direction Requirement"
        "。"
    )

else:

    display_columns = [

        "prompt_id",
        "requirement_index",
        "required_turn",
        "human_label",
        "initial_yaw_deg",
        "final_yaw_deg",
        "net_rotation_deg",
        "total_rotation_deg",
        "raw_turn_direction",
    ]

    display(
        turn_evidence_df[
            display_columns
        ]
    )


print("=" * 100)
print(
    "STEP 10H — Turn Direction Evidence collected ✅"
)
print("=" * 100)



## STEP 10I — Candidate Rotation Threshold Search

Use only Human `PASS` and `FAIL` cases for calibration.

Convert net rotation into rotation toward the required direction:

\[
R_{\text{req}}=
\begin{cases}
R_{\text{net}} & \text{required = right}\\
-R_{\text{net}} & \text{required = left}
\end{cases}
\]

Candidate rule:

\[
PASS \iff R_{\text{req}} \ge \tau_{\text{turn}}
\]

Candidate thresholds are compared against Human Gold. Because the current calibration set contains no determinate FAIL turn cases, the result must not be described as an optimal threshold.


In [ ]:
# ============================================================
# STEP 10I — Candidate Rotation Threshold Search
#
# English note: benchmark logic unchanged from the source notebook.
# English note: benchmark logic unchanged from the source notebook.
# ============================================================

import pandas as pd


# ------------------------------------------------------------
# 1. Candidate Rotation Threshold
# ------------------------------------------------------------

candidate_rotation_thresholds = [
    15,
    30,
    45,
    60,
    75,
    90,
    105,
    120,
]


# ------------------------------------------------------------
# English note: benchmark logic unchanged from the source notebook.
#
# English note: benchmark logic unchanged from the source notebook.
# English note: benchmark logic unchanged from the source notebook.
# ------------------------------------------------------------

calibration_df = turn_evidence_df[
    turn_evidence_df["human_label"].isin(
        ["PASS", "FAIL"]
    )
].copy()


print("========================================")
print("Turn Direction Calibration Cases")
print("========================================")

print(
    "Total usable cases:",
    len(calibration_df)
)

print(
    "PASS:",
    (calibration_df["human_label"] == "PASS").sum()
)

print(
    "FAIL:",
    (calibration_df["human_label"] == "FAIL").sum()
)

print(
    "UNCERTAIN excluded:",
    (turn_evidence_df["human_label"] == "UNCERTAIN").sum()
)

print("========================================")


# ------------------------------------------------------------
# English note: benchmark logic unchanged from the source notebook.
#
# right:
# English note: benchmark logic unchanged from the source notebook.
#
# left:
# English note: benchmark logic unchanged from the source notebook.
# ------------------------------------------------------------

def calculate_required_rotation(row):

    net_rotation = float(
        row["net_rotation_deg"]
    )

    required_turn = str(
        row["required_turn"]
    ).lower()

    if required_turn == "right":

        return net_rotation

    elif required_turn == "left":

        return -net_rotation

    else:

        raise ValueError(
            f"Unsupported turn direction: {required_turn}"
        )


calibration_df["required_rotation_deg"] = (
    calibration_df.apply(
        calculate_required_rotation,
        axis=1
    )
)


# ------------------------------------------------------------
# English note: benchmark logic unchanged from the source notebook.
# ------------------------------------------------------------

threshold_results = []


for threshold in candidate_rotation_thresholds:

    predictions = []

    for _, row in calibration_df.iterrows():

        required_rotation = float(
            row["required_rotation_deg"]
        )

        # English note: benchmark logic unchanged from the source notebook.
        # English note: benchmark logic unchanged from the source notebook.
        prediction = (
            "PASS"
            if required_rotation >= threshold
            else "FAIL"
        )

        predictions.append(
            prediction
        )


    # --------------------------------------------------------
    # English note: benchmark logic unchanged from the source notebook.
    # --------------------------------------------------------

    human_labels = (
        calibration_df["human_label"]
        .tolist()
    )

    total = len(
        human_labels
    )

    correct = sum(
        pred == human
        for pred, human
        in zip(
            predictions,
            human_labels
        )
    )

    false_pass = sum(
        pred == "PASS"
        and human == "FAIL"
        for pred, human
        in zip(
            predictions,
            human_labels
        )
    )

    false_fail = sum(
        pred == "FAIL"
        and human == "PASS"
        for pred, human
        in zip(
            predictions,
            human_labels
        )
    )

    accuracy = (
        correct / total
        if total > 0
        else 0.0
    )


    threshold_results.append({

        "threshold_deg":
            threshold,

        "accuracy":
            accuracy,

        "correct":
            correct,

        "total":
            total,

        "false_pass":
            false_pass,

        "false_fail":
            false_fail,
    })


# ------------------------------------------------------------
# English note: benchmark logic unchanged from the source notebook.
# ------------------------------------------------------------

rotation_threshold_results_df = pd.DataFrame(
    threshold_results
)


# ------------------------------------------------------------
# English note: benchmark logic unchanged from the source notebook.
# ------------------------------------------------------------

print("\nRequired Rotation Evidence")

display(
    calibration_df[
        [
            "prompt_id",
            "required_turn",
            "human_label",
            "net_rotation_deg",
            "required_rotation_deg",
        ]
    ]
)


# ------------------------------------------------------------
# English note: benchmark logic unchanged from the source notebook.
# ------------------------------------------------------------

print("\nCandidate Rotation Threshold Results")

display(
    rotation_threshold_results_df
)


print("=" * 100)
print(
    "STEP 10I — Candidate Rotation Threshold Search completed ✅"
)
print("=" * 100)



## STEP 10J — Initial Turn Direction Decision Rule

The provisional rule requires both the correct rotation sign and sufficient rotation magnitude.

For a right turn:

\[
PASS \iff R_{\text{net}}>0
\land R_{\text{net}}\ge\tau_{\text{turn}}
\]

For a left turn:

\[
PASS \iff R_{\text{net}}<0
\land -R_{\text{net}}\ge\tau_{\text{turn}}
\]

Pilot candidate:

\[
\tau_{\text{turn}}=90^\circ
\]

This is a provisional threshold. The Pilot calibration currently lacks determinate FAIL examples, so false-positive behaviour must be assessed through further validation.


In [ ]:
# ============================================================
# STEP 10J — Initial Turn Direction Decision Rule
#
# English note: benchmark logic unchanged from the source notebook.
# English note: benchmark logic unchanged from the source notebook.
#
# English note: benchmark logic unchanged from the source notebook.
# ============================================================


class TurnDirectionDecisionRule:

    def __init__(
        self,
        min_rotation_deg=90.0
    ):

        self.min_rotation_deg = float(
            min_rotation_deg
        )


    def evaluate(
        self,
        evidence,
        required_turn
    ):

        required_turn = str(
            required_turn
        ).lower()

        net_rotation = float(
            evidence["net_rotation_deg"]
        )


        # ----------------------------------------------------
        # English note: benchmark logic unchanged from the source notebook.
        # English note: benchmark logic unchanged from the source notebook.
        # ----------------------------------------------------

        if required_turn == "right":

            required_rotation = (
                net_rotation
            )

            correct_sign = (
                net_rotation > 0
            )


        elif required_turn == "left":

            required_rotation = (
                -net_rotation
            )

            correct_sign = (
                net_rotation < 0
            )


        else:

            raise ValueError(
                f"Unsupported turn direction: "
                f"{required_turn}"
            )


        # ----------------------------------------------------
        # English note: benchmark logic unchanged from the source notebook.
        # ----------------------------------------------------

        sufficient_rotation = (
            required_rotation
            >= self.min_rotation_deg
        )


        # ----------------------------------------------------
        # English note: benchmark logic unchanged from the source notebook.
        # ----------------------------------------------------

        prediction = (
            "PASS"
            if (
                correct_sign
                and sufficient_rotation
            )
            else "FAIL"
        )


        # ----------------------------------------------------
        # English note: benchmark logic unchanged from the source notebook.
        # ----------------------------------------------------

        return {

            "required_turn":
                required_turn,

            "prediction":
                prediction,

            "correct_sign":
                correct_sign,

            "required_rotation_deg":
                required_rotation,

            "min_rotation_threshold_deg":
                self.min_rotation_deg,

            "sufficient_rotation":
                sufficient_rotation,

            # English note: benchmark logic unchanged from the source notebook.
            "net_rotation_deg":
                net_rotation,

            "total_rotation_deg":
                evidence.get(
                    "total_rotation_deg"
                ),
        }


# ------------------------------------------------------------
# English note: benchmark logic unchanged from the source notebook.
# ------------------------------------------------------------

initial_turn_rule = (
    TurnDirectionDecisionRule(
        min_rotation_deg=90.0
    )
)


print(
    "TurnDirectionDecisionRule "
    "defined successfully."
)

print(
    "Initial Rotation Threshold:",
    initial_turn_rule.min_rotation_deg,
    "degrees"
)



## STEP 10K — Validate the Initial Turn Direction Rule on the Current Model

Apply the provisional rule to the current STEP 4 model without changing the threshold.

Compare Human Gold with automatic PASS/FAIL, while excluding Human `UNCERTAIN` from agreement statistics.


In [ ]:
# ============================================================
# STEP 10K — Current Model Pilot Validation
#
# English note: benchmark logic unchanged from the source notebook.
# English note: benchmark logic unchanged from the source notebook.
#
# English note: benchmark logic unchanged from the source notebook.
# ============================================================

import pandas as pd


validation_records = []


# ------------------------------------------------------------
# English note: benchmark logic unchanged from the source notebook.
# ------------------------------------------------------------

for _, row in turn_evidence_df.iterrows():

    # English note: benchmark logic unchanged from the source notebook.
    evidence = {

        "net_rotation_deg":
            float(row["net_rotation_deg"]),

        "total_rotation_deg":
            float(row["total_rotation_deg"]),
    }


    # --------------------------------------------------------
    # English note: benchmark logic unchanged from the source notebook.
    # --------------------------------------------------------

    result = initial_turn_rule.evaluate(

        evidence=evidence,

        required_turn=row["required_turn"],
    )


    human_label = row["human_label"]

    prediction = result["prediction"]


    # --------------------------------------------------------
    # English note: benchmark logic unchanged from the source notebook.
    #
    # English note: benchmark logic unchanged from the source notebook.
    # --------------------------------------------------------

    if human_label in ["PASS", "FAIL"]:

        match = (
            prediction == human_label
        )

    else:

        match = None


    # --------------------------------------------------------
    # English note: benchmark logic unchanged from the source notebook.
    # --------------------------------------------------------

    validation_records.append({

        "prompt_id":
            row["prompt_id"],

        "required_turn":
            row["required_turn"],

        "human_label":
            human_label,

        "net_rotation_deg":
            row["net_rotation_deg"],

        "required_rotation_deg":
            result["required_rotation_deg"],

        "threshold_deg":
            result["min_rotation_threshold_deg"],

        "correct_sign":
            result["correct_sign"],

        "sufficient_rotation":
            result["sufficient_rotation"],

        "prediction":
            prediction,

        "match":
            match,
    })


# ------------------------------------------------------------
# English note: benchmark logic unchanged from the source notebook.
# ------------------------------------------------------------

turn_validation_df = pd.DataFrame(
    validation_records
)


# ------------------------------------------------------------
# English note: benchmark logic unchanged from the source notebook.
# ------------------------------------------------------------

print("=" * 100)
print(
    "STEP 10K — Current Model "
    "Turn Direction Validation"
)
print("=" * 100)


display(
    turn_validation_df
)


# ------------------------------------------------------------
# English note: benchmark logic unchanged from the source notebook.
# ------------------------------------------------------------

scored_df = turn_validation_df[
    turn_validation_df["human_label"].isin(
        ["PASS", "FAIL"]
    )
].copy()


total = len(
    scored_df
)

correct = int(
    scored_df["match"].sum()
)

accuracy = (
    correct / total
    if total > 0
    else 0.0
)


# ------------------------------------------------------------
# 8. False PASS / False FAIL
# ------------------------------------------------------------

false_pass = int(
    (
        (scored_df["prediction"] == "PASS")
        &
        (scored_df["human_label"] == "FAIL")
    ).sum()
)


false_fail = int(
    (
        (scored_df["prediction"] == "FAIL")
        &
        (scored_df["human_label"] == "PASS")
    ).sum()
)


# ------------------------------------------------------------
# 9. Summary
# ------------------------------------------------------------

print("----------------------------------------")
print("Validation Summary")
print("----------------------------------------")

print(
    "Scored cases       :",
    total
)

print(
    "Correct            :",
    correct
)

print(
    f"Accuracy           : {accuracy:.3f}"
)

print(
    "False PASS         :",
    false_pass
)

print(
    "False FAIL         :",
    false_fail
)

print(
    "UNCERTAIN excluded :",
    (
        turn_validation_df["human_label"]
        == "UNCERTAIN"
    ).sum()
)


print("=" * 100)
print(
    "STEP 10K — Current Model Validation completed ✅"
)
print("=" * 100)



## STEP 10L — Cross-model Turn Direction Validation (run later)

The current-model validation is complete.  
Apply the same frozen rule to other models after the remaining requirement evaluators are available.


# Phase C — Action / Event Requirement Evaluation

## STEP 10M — Final Action Evaluation Strategy: Event Detector Registry

TMR++ was investigated as a single generic semantic evaluator for Action requirements. The Pilot tested:

- absolute TMR++ similarity,
- contrastive action ranking,
- prompt ensembles,
- sliding-window TMR++ similarity.

These formulations did not reliably separate requirement-level Human `PASS` and `FAIL`. Therefore, TMR++ similarity is not used as the final Action PASS/FAIL rule.

Instead, Action requirements are routed to action/event-family detectors. Their event evidence can later be reused by generic Count, Order, and Simultaneous evaluators.

| Action / Event family | Detector |
|---|---|
| walk / locomotion | `LocomotionEventDetector` |
| turn | `TurnEventDetector` |
| jump | `JumpEventDetector` |
| raise hand(s) | `ArmRaiseEventDetector` |
| kick | `KickEventDetector` |
| reach | `ReachEventDetector` |


### STEP 10AA — TurnEventDetector

Use the yaw trajectory from `RotationEvaluator` to extract the temporal location of turn events.

The detector should provide event-level evidence:

- `start_frame`
- `end_frame`
- `peak_frame`
- `start_time`
- `end_time`
- `signed_rotation_deg`
- `absolute_rotation_deg`
- `direction`

This timing evidence can later support Action, Turn Direction, Count, Temporal Order, and Simultaneous requirements.

At this stage, first extract candidate events and compare them with Human Gold before freezing a final benchmark rule.


In [ ]:
# ============================================================
# English note: benchmark logic unchanged from the source notebook.
#
# English note: benchmark logic unchanged from the source notebook.
# English note: benchmark logic unchanged from the source notebook.
#
# Test:
#   C1-07
#   A person turns to the right.
# ============================================================

import numpy as np
from pathlib import Path


# ------------------------------------------------------------
# 1. Test Motion
# ------------------------------------------------------------

TEST_PROMPT_ID = "C1-07"

motion_path = (
    Path(MODEL_INPUT_DIR)
    / f"{TEST_PROMPT_ID}.npy"
)

motion = np.load(
    motion_path
).astype(np.float32)


print("=" * 90)
print("STEP 10AA-1 — Yaw Trajectory Sanity Check")
print("=" * 90)

print(f"Prompt ID    : {TEST_PROMPT_ID}")
print(f"Motion Shape : {motion.shape}")


# ------------------------------------------------------------
# 2. HumanML3D-style Hip Joints
#
# Joint 1 = Left Hip
# Joint 2 = Right Hip
# ------------------------------------------------------------

left_hip = motion[:, 1, :]
right_hip = motion[:, 2, :]


# ------------------------------------------------------------
# 3. Left → Right Hip Vector
# ------------------------------------------------------------

lateral = (
    right_hip
    -
    left_hip
)

lateral_xz = lateral[:, [0, 2]]


# ------------------------------------------------------------
# 4. Estimate Forward Heading
#
# Standardised coordinate:
#   +X = Right
#   +Z = Forward
#
# English note: benchmark logic unchanged from the source notebook.
# English note: benchmark logic unchanged from the source notebook.
# ------------------------------------------------------------

heading = np.stack(
    [
        -lateral_xz[:, 1],
        lateral_xz[:, 0],
    ],
    axis=1,
)


# ------------------------------------------------------------
# 5. Calculate Yaw
#
# yaw = atan2(X, Z)
#
# Positive:
#   Right turn
#
# Negative:
#   Left turn
# ------------------------------------------------------------

yaw_rad = np.arctan2(
    heading[:, 0],
    heading[:, 1],
)

yaw_rad = np.unwrap(
    yaw_rad
)

yaw_deg = np.degrees(
    yaw_rad
)


# ------------------------------------------------------------
# 6. Normalise Initial Yaw to 0°
#
# English note: benchmark logic unchanged from the source notebook.
# English note: benchmark logic unchanged from the source notebook.
# ------------------------------------------------------------

relative_yaw_deg = (
    yaw_deg
    -
    yaw_deg[0]
)


# ------------------------------------------------------------
# 7. Frame-to-Frame Yaw Change
# ------------------------------------------------------------

yaw_delta_deg = np.diff(
    relative_yaw_deg,
    prepend=relative_yaw_deg[0],
)


# ------------------------------------------------------------
# 8. Basic Summary
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("YAW SUMMARY")
print("=" * 90)

print(
    f"Initial Yaw       : "
    f"{relative_yaw_deg[0]:.3f}°"
)

print(
    f"Final Yaw         : "
    f"{relative_yaw_deg[-1]:.3f}°"
)

print(
    f"Net Rotation      : "
    f"{relative_yaw_deg[-1] - relative_yaw_deg[0]:.3f}°"
)

print(
    f"Maximum Yaw       : "
    f"{np.max(relative_yaw_deg):.3f}°"
)

print(
    f"Minimum Yaw       : "
    f"{np.min(relative_yaw_deg):.3f}°"
)

print(
    f"Total Abs Rotation: "
    f"{np.sum(np.abs(yaw_delta_deg)):.3f}°"
)


# ------------------------------------------------------------
# 9. Print Yaw Every 5 Frames
#
# English note: benchmark logic unchanged from the source notebook.
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("YAW TRAJECTORY — EVERY 5 FRAMES")
print("=" * 90)

for frame in range(
    0,
    len(relative_yaw_deg),
    5,
):

    print(
        f"Frame {frame:3d} | "
        f"Time {frame / 20:5.2f}s | "
        f"Yaw {relative_yaw_deg[frame]:8.3f}° | "
        f"ΔYaw {yaw_delta_deg[frame]:8.3f}°"
    )


# ------------------------------------------------------------
# 10. Largest Frame-to-Frame Rotation
# ------------------------------------------------------------

largest_change_frame = int(
    np.argmax(
        np.abs(yaw_delta_deg)
    )
)

print("\n" + "=" * 90)
print("LARGEST FRAME-TO-FRAME YAW CHANGE")
print("=" * 90)

print(
    f"Frame       : "
    f"{largest_change_frame}"
)

print(
    f"Time        : "
    f"{largest_change_frame / 20:.2f} sec"
)

print(
    f"Yaw Change  : "
    f"{yaw_delta_deg[largest_change_frame]:.3f}°"
)

print(
    f"Current Yaw : "
    f"{relative_yaw_deg[largest_change_frame]:.3f}°"
)


print("\n" + "=" * 90)
print("STEP 10AA-1 completed.")
print("No Turn Event threshold or PASS / FAIL rule was applied.")
print("=" * 90)


### STEP 10AA-2 — Turn Activity Evidence

Frame-to-frame yaw can contain short pauses and small opposite-direction changes. Direct segmentation may therefore split one human-level turn into several events.

Compute frame-to-frame yaw change:

\[
\Delta\theta_t=\tilde{\theta}_t-\tilde{\theta}_{t-1}
\]

and smooth it with a 5-frame moving average:

\[
\bar{\Delta\theta}_t=
\frac{1}{5}\sum_{k=-2}^{2}\Delta\theta_{t+k}
\]

Use this only as temporal Turn Activity evidence at this stage; it is not yet a final PASS/FAIL threshold.


In [ ]:
# ============================================================
# STEP 10AA-2 — Turn Activity Evidence
#
# English note: benchmark logic unchanged from the source notebook.
# English note: benchmark logic unchanged from the source notebook.
#
# IMPORTANT:
# English note: benchmark logic unchanged from the source notebook.
# English note: benchmark logic unchanged from the source notebook.
# English note: benchmark logic unchanged from the source notebook.
# ============================================================

import numpy as np


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

SMOOTHING_WINDOW = 5


# ------------------------------------------------------------
# 2. Moving Average of Frame-to-Frame Yaw Change
# ------------------------------------------------------------

kernel = np.ones(
    SMOOTHING_WINDOW,
    dtype=np.float32
) / SMOOTHING_WINDOW


smoothed_yaw_delta = np.convolve(
    yaw_delta_deg,
    kernel,
    mode="same",
)


# ------------------------------------------------------------
# 3. Absolute Turn Activity
#
# English note: benchmark logic unchanged from the source notebook.
# English note: benchmark logic unchanged from the source notebook.
# ------------------------------------------------------------

turn_activity = np.abs(
    smoothed_yaw_delta
)


# ------------------------------------------------------------
# 4. Basic Summary
# ------------------------------------------------------------

print("=" * 100)
print("STEP 10AA-2 — Turn Activity Evidence")
print("=" * 100)

print(
    f"Smoothing Window : "
    f"{SMOOTHING_WINDOW} frames "
    f"({SMOOTHING_WINDOW / 20:.2f} sec)"
)

print(
    f"Max Raw ΔYaw     : "
    f"{np.max(np.abs(yaw_delta_deg)):.3f}°/frame"
)

print(
    f"Max Smoothed ΔYaw: "
    f"{np.max(turn_activity):.3f}°/frame"
)


# ------------------------------------------------------------
# 5. Maximum Smoothed Activity
# ------------------------------------------------------------

max_activity_frame = int(
    np.argmax(turn_activity)
)

print("\n" + "=" * 100)
print("MAXIMUM TURN ACTIVITY")
print("=" * 100)

print(
    f"Frame            : "
    f"{max_activity_frame}"
)

print(
    f"Time             : "
    f"{max_activity_frame / 20:.2f} sec"
)

print(
    f"Relative Yaw     : "
    f"{relative_yaw_deg[max_activity_frame]:.3f}°"
)

print(
    f"Raw ΔYaw         : "
    f"{yaw_delta_deg[max_activity_frame]:.3f}°"
)

print(
    f"Smoothed ΔYaw    : "
    f"{smoothed_yaw_delta[max_activity_frame]:.3f}°"
)


# ------------------------------------------------------------
# 6. Print Evidence Every 2 Frames
#
# English note: benchmark logic unchanged from the source notebook.
# English note: benchmark logic unchanged from the source notebook.
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("TURN ACTIVITY TRAJECTORY — EVERY 2 FRAMES")
print("=" * 100)

print(
    f"{'Frame':>6} | "
    f"{'Time':>6} | "
    f"{'Yaw':>10} | "
    f"{'Raw Δ':>10} | "
    f"{'Smooth Δ':>10} | "
    f"{'Activity':>10}"
)

print("-" * 100)


for frame in range(
    0,
    len(relative_yaw_deg),
    2,
):

    print(
        f"{frame:6d} | "
        f"{frame / 20:5.2f}s | "
        f"{relative_yaw_deg[frame]:9.3f}° | "
        f"{yaw_delta_deg[frame]:9.3f}° | "
        f"{smoothed_yaw_delta[frame]:9.3f}° | "
        f"{turn_activity[frame]:9.3f}"
    )


# ------------------------------------------------------------
# 7. Top Activity Frames
# ------------------------------------------------------------

TOP_K = 15

top_frames = np.argsort(
    turn_activity
)[::-1][:TOP_K]


print("\n" + "=" * 100)
print(f"TOP {TOP_K} TURN-ACTIVITY FRAMES")
print("=" * 100)

for rank, frame in enumerate(
    top_frames,
    start=1,
):

    direction = (
        "right"
        if smoothed_yaw_delta[frame] > 0
        else "left"
        if smoothed_yaw_delta[frame] < 0
        else "none"
    )

    print(
        f"{rank:2d}. "
        f"Frame {frame:3d} | "
        f"{frame / 20:5.2f}s | "
        f"Yaw {relative_yaw_deg[frame]:8.3f}° | "
        f"Smooth Δ {smoothed_yaw_delta[frame]:8.3f}° | "
        f"{direction}"
    )


# ------------------------------------------------------------
# 8. Finish
# ------------------------------------------------------------

print("\n" + "=" * 100)

print(
    "STEP 10AA-2 completed."
)

print(
    "Turn activity evidence only."
)

print(
    "No event boundary, count, threshold, "
    "or PASS / FAIL rule was applied."
)

print("=" * 100)


### STEP 10AA-3 — Candidate Turn Event Segmentation

Candidate segmentation procedure:

1. Extract active turn frames from smoothed yaw change.
2. Convert consecutive active frames into candidate segments.
3. Merge same-direction segments separated by a short gap.
4. Measure each candidate event's rotation and temporal interval.

Candidate parameters:

\[
|\bar{\Delta\theta}_t|\ge0.5^\circ/\text{frame}
\]

and

\[
\text{maximum merge gap}=8\text{ frames}=0.4\text{ s at 20 fps}
\]

These are candidate parameters, not final benchmark thresholds.


In [ ]:
# ============================================================
# STEP 10AA-3 — Candidate Turn Event Segmentation
#
# English note: benchmark logic unchanged from the source notebook.
#
# IMPORTANT:
# English note: benchmark logic unchanged from the source notebook.
# English note: benchmark logic unchanged from the source notebook.
# English note: benchmark logic unchanged from the source notebook.
# ============================================================

import numpy as np


# ============================================================
# 1. Candidate Parameters
# ============================================================

ACTIVITY_THRESHOLD = 0.5       # deg / frame
MAX_MERGE_GAP = 8             # frames
FPS = 20


print("=" * 100)
print("STEP 10AA-3 — Candidate Turn Event Segmentation")
print("=" * 100)

print(
    f"Activity Threshold : "
    f"{ACTIVITY_THRESHOLD:.2f}°/frame"
)

print(
    f"Maximum Merge Gap  : "
    f"{MAX_MERGE_GAP} frames "
    f"({MAX_MERGE_GAP / FPS:.2f} sec)"
)


# ============================================================
# 2. Active Turn Frames
#
# English note: benchmark logic unchanged from the source notebook.
# English note: benchmark logic unchanged from the source notebook.
# ============================================================

active_mask = (
    np.abs(smoothed_yaw_delta)
    >= ACTIVITY_THRESHOLD
)

active_frames = np.where(
    active_mask
)[0]


print("\n" + "=" * 100)
print("ACTIVE TURN FRAMES")
print("=" * 100)

print(
    f"Number of active frames : "
    f"{len(active_frames)}"
)

if len(active_frames) > 0:

    print(
        f"First active frame      : "
        f"{active_frames[0]}"
    )

    print(
        f"Last active frame       : "
        f"{active_frames[-1]}"
    )


# ============================================================
# 3. Convert Active Frames into Consecutive Segments
# ============================================================

raw_segments = []


if len(active_frames) > 0:

    segment_start = int(
        active_frames[0]
    )

    previous_frame = int(
        active_frames[0]
    )


    for frame in active_frames[1:]:

        frame = int(frame)

        # English note: benchmark logic unchanged from the source notebook.
        if frame == previous_frame + 1:

            previous_frame = frame

            continue


        # English note: benchmark logic unchanged from the source notebook.
        raw_segments.append(
            {
                "start_frame": segment_start,
                "end_frame": previous_frame,
            }
        )


        # English note: benchmark logic unchanged from the source notebook.
        segment_start = frame
        previous_frame = frame


    # English note: benchmark logic unchanged from the source notebook.
    raw_segments.append(
        {
            "start_frame": segment_start,
            "end_frame": previous_frame,
        }
    )


# ============================================================
# 4. Add Direction / Rotation Evidence
# ============================================================

for segment in raw_segments:

    start = segment["start_frame"]
    end = segment["end_frame"]

    signed_rotation = (
        relative_yaw_deg[end]
        -
        relative_yaw_deg[start]
    )

    segment["signed_rotation_deg"] = float(
        signed_rotation
    )

    segment["direction"] = (
        "right"
        if signed_rotation > 0
        else "left"
        if signed_rotation < 0
        else "none"
    )


# ============================================================
# 5. Display Raw Segments
# ============================================================

print("\n" + "=" * 100)
print("RAW CANDIDATE SEGMENTS")
print("=" * 100)


if len(raw_segments) == 0:

    print(
        "No candidate turn segments detected."
    )


else:

    for i, segment in enumerate(
        raw_segments,
        start=1,
    ):

        start = segment["start_frame"]
        end = segment["end_frame"]

        print(
            f"[{i}] "
            f"Frames {start:3d}–{end:3d} | "
            f"{start / FPS:5.2f}–"
            f"{end / FPS:5.2f}s | "
            f"{segment['direction']:5s} | "
            f"Rotation "
            f"{segment['signed_rotation_deg']:+8.3f}°"
        )


# ============================================================
# 6. Merge Short Gaps
#
# English note: benchmark logic unchanged from the source notebook.
#
# 1. Gap <= MAX_MERGE_GAP
# English note: benchmark logic unchanged from the source notebook.
#
# ============================================================

merged_segments = []


for segment in raw_segments:

    if not merged_segments:

        merged_segments.append(
            segment.copy()
        )

        continue


    previous = merged_segments[-1]

    gap = (
        segment["start_frame"]
        -
        previous["end_frame"]
        -
        1
    )


    same_direction = (
        segment["direction"]
        ==
        previous["direction"]
    )


    if (
        gap <= MAX_MERGE_GAP
        and
        same_direction
    ):

        # ----------------------------------------
        # Merge
        # ----------------------------------------

        previous["end_frame"] = (
            segment["end_frame"]
        )

        start = previous["start_frame"]
        end = previous["end_frame"]

        signed_rotation = (
            relative_yaw_deg[end]
            -
            relative_yaw_deg[start]
        )

        previous["signed_rotation_deg"] = float(
            signed_rotation
        )

        previous["direction"] = (
            "right"
            if signed_rotation > 0
            else "left"
            if signed_rotation < 0
            else "none"
        )


    else:

        merged_segments.append(
            segment.copy()
        )


# ============================================================
# 7. Add Final Event Evidence
# ============================================================

for segment in merged_segments:

    start = segment["start_frame"]
    end = segment["end_frame"]

    segment["start_time"] = (
        start / FPS
    )

    segment["end_time"] = (
        end / FPS
    )

    segment["duration_frames"] = (
        end - start + 1
    )

    segment["duration_sec"] = (
        segment["duration_frames"]
        / FPS
    )

    segment["absolute_rotation_deg"] = abs(
        segment["signed_rotation_deg"]
    )


    # ----------------------------------------
    # Peak Turn Activity
    # ----------------------------------------

    local_activity = turn_activity[
        start:end + 1
    ]

    peak_local_index = int(
        np.argmax(local_activity)
    )

    peak_frame = (
        start
        +
        peak_local_index
    )

    segment["peak_frame"] = (
        peak_frame
    )

    segment["peak_time"] = (
        peak_frame / FPS
    )

    segment["peak_activity"] = float(
        turn_activity[peak_frame]
    )


# ============================================================
# 8. Display Merged Candidate Events
# ============================================================

print("\n" + "=" * 100)
print("MERGED CANDIDATE TURN EVENTS")
print("=" * 100)


if len(merged_segments) == 0:

    print(
        "No candidate turn events detected."
    )


else:

    for i, event in enumerate(
        merged_segments,
        start=1,
    ):

        print(f"\nCandidate Event {i}")

        print(
            f"  Frames            : "
            f"{event['start_frame']}–"
            f"{event['end_frame']}"
        )

        print(
            f"  Time              : "
            f"{event['start_time']:.2f}–"
            f"{event['end_time']:.2f} sec"
        )

        print(
            f"  Duration          : "
            f"{event['duration_frames']} frames "
            f"({event['duration_sec']:.2f} sec)"
        )

        print(
            f"  Direction         : "
            f"{event['direction']}"
        )

        print(
            f"  Signed Rotation   : "
            f"{event['signed_rotation_deg']:+.3f}°"
        )

        print(
            f"  Absolute Rotation : "
            f"{event['absolute_rotation_deg']:.3f}°"
        )

        print(
            f"  Peak Frame        : "
            f"{event['peak_frame']}"
        )

        print(
            f"  Peak Time         : "
            f"{event['peak_time']:.2f} sec"
        )

        print(
            f"  Peak Activity     : "
            f"{event['peak_activity']:.3f}°/frame"
        )


# ============================================================
# 9. Finish
# ============================================================

print("\n" + "=" * 100)

print(
    f"Raw Segments    : "
    f"{len(raw_segments)}"
)

print(
    f"Merged Events   : "
    f"{len(merged_segments)}"
)

print(
    "STEP 10AA-3 completed."
)

print(
    "Candidate segmentation only."
)

print(
    "No final Turn Event threshold, count, "
    "Order, or PASS / FAIL rule was applied."
)

print("=" * 100)


### STEP 10AA-4 — Turn Candidate Events Across Pilot Cases

Apply the same candidate segmentation to every Pilot motion containing a Turn Direction requirement.

For each candidate event, record:

- start/end frame and time,
- duration,
- direction,
- signed/absolute rotation,
- peak activity.

Human `UNCERTAIN` cases are kept as evidence but excluded from calibration.


In [ ]:
# ============================================================
# STEP 10AA-4 — Turn Candidate Events Across Pilot Cases
#
# English note: benchmark logic unchanged from the source notebook.
# English note: benchmark logic unchanged from the source notebook.
#
# IMPORTANT:
# English note: benchmark logic unchanged from the source notebook.
# English note: benchmark logic unchanged from the source notebook.
# English note: benchmark logic unchanged from the source notebook.
# English note: benchmark logic unchanged from the source notebook.
# ============================================================

import numpy as np
from pathlib import Path


# ============================================================
# 1. Configuration
# ============================================================

FPS = 20

SMOOTHING_WINDOW = 5

ACTIVITY_THRESHOLD = 0.5   # deg / frame
MAX_MERGE_GAP = 8          # frames


TURN_PROMPT_IDS = [
    "C1-06",
    "C1-07",
    "C4-01",
    "C4-02",
    "C5-16",
]


# ============================================================
# 2. Yaw Trajectory Extraction
# ============================================================

def extract_yaw_evidence(motion):
    """English documentation: this function/class preserves the original benchmark logic. See parameter names, comments, and returned fields."""

    left_hip = motion[:, 1, :]
    right_hip = motion[:, 2, :]

    lateral = (
        right_hip
        -
        left_hip
    )

    lateral_xz = lateral[:, [0, 2]]


    # --------------------------------------------------------
    # Body Forward Heading
    # --------------------------------------------------------

    heading = np.stack(
        [
            -lateral_xz[:, 1],
            lateral_xz[:, 0],
        ],
        axis=1,
    )


    # --------------------------------------------------------
    # Yaw
    # --------------------------------------------------------

    yaw_rad = np.arctan2(
        heading[:, 0],
        heading[:, 1],
    )

    yaw_rad = np.unwrap(
        yaw_rad
    )

    yaw_deg = np.degrees(
        yaw_rad
    )


    # --------------------------------------------------------
    # Initial heading = 0°
    # --------------------------------------------------------

    relative_yaw_deg = (
        yaw_deg
        -
        yaw_deg[0]
    )


    # --------------------------------------------------------
    # Frame-to-frame Yaw Change
    # --------------------------------------------------------

    yaw_delta_deg = np.diff(
        relative_yaw_deg,
        prepend=relative_yaw_deg[0],
    )


    # --------------------------------------------------------
    # Smooth Yaw Change
    # --------------------------------------------------------

    kernel = (
        np.ones(
            SMOOTHING_WINDOW,
            dtype=np.float32,
        )
        /
        SMOOTHING_WINDOW
    )

    smoothed_yaw_delta = np.convolve(
        yaw_delta_deg,
        kernel,
        mode="same",
    )

    turn_activity = np.abs(
        smoothed_yaw_delta
    )


    return {
        "relative_yaw_deg": relative_yaw_deg,
        "yaw_delta_deg": yaw_delta_deg,
        "smoothed_yaw_delta": smoothed_yaw_delta,
        "turn_activity": turn_activity,
    }


# ============================================================
# 3. Candidate Turn Event Extraction
# ============================================================

def extract_candidate_turn_events(
    motion,
):
    """English documentation: this function/class preserves the original benchmark logic. See parameter names, comments, and returned fields."""

    yaw_evidence = extract_yaw_evidence(
        motion
    )

    relative_yaw_deg = (
        yaw_evidence["relative_yaw_deg"]
    )

    smoothed_yaw_delta = (
        yaw_evidence["smoothed_yaw_delta"]
    )

    turn_activity = (
        yaw_evidence["turn_activity"]
    )


    # --------------------------------------------------------
    # Active Frames
    # --------------------------------------------------------

    active_mask = (
        np.abs(smoothed_yaw_delta)
        >= ACTIVITY_THRESHOLD
    )

    active_frames = np.where(
        active_mask
    )[0]


    # --------------------------------------------------------
    # Consecutive Segments
    # --------------------------------------------------------

    raw_segments = []


    if len(active_frames) > 0:

        segment_start = int(
            active_frames[0]
        )

        previous_frame = int(
            active_frames[0]
        )


        for frame in active_frames[1:]:

            frame = int(frame)


            if frame == previous_frame + 1:

                previous_frame = frame
                continue


            raw_segments.append(
                {
                    "start_frame": segment_start,
                    "end_frame": previous_frame,
                }
            )


            segment_start = frame
            previous_frame = frame


        raw_segments.append(
            {
                "start_frame": segment_start,
                "end_frame": previous_frame,
            }
        )


    # --------------------------------------------------------
    # Direction Evidence
    # --------------------------------------------------------

    for segment in raw_segments:

        start = segment["start_frame"]
        end = segment["end_frame"]

        signed_rotation = (
            relative_yaw_deg[end]
            -
            relative_yaw_deg[start]
        )

        segment["signed_rotation_deg"] = float(
            signed_rotation
        )

        segment["direction"] = (
            "right"
            if signed_rotation > 0
            else "left"
            if signed_rotation < 0
            else "none"
        )


    # --------------------------------------------------------
    # Merge Same-direction Segments
    # --------------------------------------------------------

    merged_segments = []


    for segment in raw_segments:

        if not merged_segments:

            merged_segments.append(
                segment.copy()
            )

            continue


        previous = merged_segments[-1]

        gap = (
            segment["start_frame"]
            -
            previous["end_frame"]
            -
            1
        )

        same_direction = (
            segment["direction"]
            ==
            previous["direction"]
        )


        if (
            gap <= MAX_MERGE_GAP
            and
            same_direction
        ):

            previous["end_frame"] = (
                segment["end_frame"]
            )

            start = previous["start_frame"]
            end = previous["end_frame"]

            signed_rotation = (
                relative_yaw_deg[end]
                -
                relative_yaw_deg[start]
            )

            previous["signed_rotation_deg"] = float(
                signed_rotation
            )

            previous["direction"] = (
                "right"
                if signed_rotation > 0
                else "left"
                if signed_rotation < 0
                else "none"
            )


        else:

            merged_segments.append(
                segment.copy()
            )


    # --------------------------------------------------------
    # Final Candidate Evidence
    # --------------------------------------------------------

    for event in merged_segments:

        start = event["start_frame"]
        end = event["end_frame"]

        event["start_time"] = (
            start / FPS
        )

        event["end_time"] = (
            end / FPS
        )

        event["duration_frames"] = (
            end - start + 1
        )

        event["duration_sec"] = (
            event["duration_frames"]
            / FPS
        )

        event["absolute_rotation_deg"] = abs(
            event["signed_rotation_deg"]
        )


        # ----------------------------------------------------
        # Peak Activity
        # ----------------------------------------------------

        local_activity = (
            turn_activity[start:end + 1]
        )

        peak_local_index = int(
            np.argmax(local_activity)
        )

        peak_frame = (
            start
            +
            peak_local_index
        )

        event["peak_frame"] = (
            peak_frame
        )

        event["peak_time"] = (
            peak_frame / FPS
        )

        event["peak_activity"] = float(
            turn_activity[peak_frame]
        )


    return {
        "raw_segments": raw_segments,
        "candidate_events": merged_segments,
        "yaw_evidence": yaw_evidence,
    }


# ============================================================
# 4. Find Turn Direction Requirement
# ============================================================

def find_turn_requirement(case):
    """English documentation: this function/class preserves the original benchmark logic. See parameter names, comments, and returned fields."""

    for index, requirement in enumerate(
        case["requirements"]
    ):

        if requirement.get("type") == "turn_direction":

            expected = requirement.get(
                "value",
                requirement.get("expected"),
            )

            return (
                index,
                expected,
            )


    return (
        None,
        None,
    )


# ============================================================
# 5. Pilot Cases
# ============================================================

case_lookup = {
    (
        case.get("prompt_id")
        or case.get("id")
        or case.get("case_id")
    ): case

    for case in pilot_prompts
}


all_turn_results = []


print("=" * 110)
print("STEP 10AA-4 — TURN CANDIDATE EVENTS ACROSS PILOT CASES")
print("=" * 110)

print(
    f"Activity Threshold : "
    f"{ACTIVITY_THRESHOLD:.2f}°/frame"
)

print(
    f"Merge Gap          : "
    f"{MAX_MERGE_GAP} frames "
    f"({MAX_MERGE_GAP / FPS:.2f}s)"
)

print(
    f"Smoothing Window   : "
    f"{SMOOTHING_WINDOW} frames"
)


# ============================================================
# 6. Evaluate Each Turn Case
# ============================================================

for prompt_id in TURN_PROMPT_IDS:

    print("\n" + "=" * 110)


    # --------------------------------------------------------
    # Benchmark Case
    # --------------------------------------------------------

    if prompt_id not in case_lookup:

        print(
            f"{prompt_id}: "
            f"Case not found in pilot_prompts."
        )

        continue


    case = case_lookup[prompt_id]


    # --------------------------------------------------------
    # Turn Requirement
    # --------------------------------------------------------

    (
        requirement_index,
        expected_direction,
    ) = find_turn_requirement(
        case
    )


    if requirement_index is None:

        print(
            f"{prompt_id}: "
            f"No turn_direction requirement found."
        )

        continue


    # --------------------------------------------------------
    # Human Gold
    # --------------------------------------------------------

    human_label = get_human_label(
        case,
        requirement_index,
    )


    # --------------------------------------------------------
    # Motion
    # --------------------------------------------------------

    motion_path = (
        Path(MODEL_INPUT_DIR)
        /
        f"{prompt_id}.npy"
    )


    if not motion_path.exists():

        print(
            f"{prompt_id}: "
            f"Motion file not found:"
        )

        print(
            motion_path
        )

        continue


    motion = np.load(
        motion_path
    ).astype(np.float32)


    # --------------------------------------------------------
    # Candidate Events
    # --------------------------------------------------------

    result = extract_candidate_turn_events(
        motion
    )

    raw_segments = (
        result["raw_segments"]
    )

    candidate_events = (
        result["candidate_events"]
    )


    # --------------------------------------------------------
    # Case Information
    # --------------------------------------------------------

    prompt_text = case.get(
        "prompt",
        case.get(
            "text",
            "",
        ),
    )


    print(
        f"Prompt ID          : "
        f"{prompt_id}"
    )

    print(
        f"Prompt             : "
        f"{prompt_text}"
    )

    print(
        f"Expected Direction : "
        f"{expected_direction}"
    )

    print(
        f"Human Gold         : "
        f"{human_label}"
    )

    print(
        f"Motion Shape       : "
        f"{motion.shape}"
    )

    print(
        f"Raw Segments       : "
        f"{len(raw_segments)}"
    )

    print(
        f"Merged Candidates  : "
        f"{len(candidate_events)}"
    )


    # --------------------------------------------------------
    # Candidate Event Details
    # --------------------------------------------------------

    if len(candidate_events) == 0:

        print(
            "\nNo Candidate Turn Event."
        )


    else:

        print(
            "\nCandidate Turn Events:"
        )


        for event_index, event in enumerate(
            candidate_events,
            start=1,
        ):

            print(
                f"\n  [{event_index}]"
            )

            print(
                f"      Frames        : "
                f"{event['start_frame']}–"
                f"{event['end_frame']}"
            )

            print(
                f"      Time          : "
                f"{event['start_time']:.2f}–"
                f"{event['end_time']:.2f}s"
            )

            print(
                f"      Duration      : "
                f"{event['duration_sec']:.2f}s"
            )

            print(
                f"      Direction     : "
                f"{event['direction']}"
            )

            print(
                f"      Rotation      : "
                f"{event['signed_rotation_deg']:+.3f}°"
            )

            print(
                f"      Abs Rotation  : "
                f"{event['absolute_rotation_deg']:.3f}°"
            )

            print(
                f"      Peak Frame    : "
                f"{event['peak_frame']}"
            )

            print(
                f"      Peak Activity : "
                f"{event['peak_activity']:.3f}°/frame"
            )


    # --------------------------------------------------------
    # Save Result
    # --------------------------------------------------------

    all_turn_results.append(
        {
            "prompt_id": prompt_id,
            "prompt": prompt_text,
            "expected_direction": expected_direction,
            "human_label": human_label,
            "raw_segment_count": len(
                raw_segments
            ),
            "candidate_event_count": len(
                candidate_events
            ),
            "candidate_events": candidate_events,
        }
    )


# ============================================================
# 7. Compact Summary
# ============================================================

print("\n" + "=" * 110)
print("COMPACT SUMMARY")
print("=" * 110)


for result in all_turn_results:

    print(
        f"\n{result['prompt_id']} | "
        f"Expected={result['expected_direction']} | "
        f"Human={result['human_label']} | "
        f"Candidates={result['candidate_event_count']}"
    )


    for i, event in enumerate(
        result["candidate_events"],
        start=1,
    ):

        print(
            f"    [{i}] "
            f"{event['direction']:5s} | "
            f"{event['signed_rotation_deg']:+8.3f}° | "
            f"{event['start_time']:.2f}–"
            f"{event['end_time']:.2f}s | "
            f"Peak={event['peak_activity']:.3f}"
        )


# ============================================================
# 8. Finish
# ============================================================

print("\n" + "=" * 110)

print(
    f"Turn cases processed : "
    f"{len(all_turn_results)}"
)

print(
    "STEP 10AA-4 completed."
)

print(
    "Evidence collection only."
)

print(
    "No final Event Validation threshold, "
    "Turn Count, Order, or PASS / FAIL rule was applied."
)

print("=" * 110)


### STEP 10AA-5 — Main Turn Candidate Selection

For a required turn direction \(d\):

1. Keep candidate events whose direction matches \(d\).
2. Select the event with the largest absolute rotation as the Main Turn Candidate:

\[
e^*=\arg\max_{e\in E_d}|R_e|
\]

3. Keep all remaining same-direction events as secondary candidates.

Diagnostics:

\[
Gap=|R_{\text{main}}|-|R_{\text{second}}|
\]

\[
Ratio=\frac{|R_{\text{main}}|}{|R_{\text{second}}|}
\]

Gap and Ratio are diagnostic evidence only, not current decision conditions.


In [ ]:
# ============================================================
# STEP 10AA-5 — Main Turn Candidate Selection
#
# English note: benchmark logic unchanged from the source notebook.
# English note: benchmark logic unchanged from the source notebook.
# English note: benchmark logic unchanged from the source notebook.
#
# IMPORTANT:
# English note: benchmark logic unchanged from the source notebook.
# English note: benchmark logic unchanged from the source notebook.
# English note: benchmark logic unchanged from the source notebook.
# ============================================================

import numpy as np


# ============================================================
# 1. Main Candidate Selection Function
# ============================================================

def select_main_turn_candidate(
    candidate_events,
    expected_direction,
):
    """English documentation: this function/class preserves the original benchmark logic. See parameter names, comments, and returned fields."""

    # --------------------------------------------------------
    # English note: benchmark logic unchanged from the source notebook.
    # --------------------------------------------------------

    same_direction_candidates = [
        event.copy()
        for event in candidate_events
        if event["direction"] == expected_direction
    ]


    # --------------------------------------------------------
    # English note: benchmark logic unchanged from the source notebook.
    # --------------------------------------------------------

    same_direction_candidates.sort(
        key=lambda event: event["absolute_rotation_deg"],
        reverse=True,
    )


    # --------------------------------------------------------
    # English note: benchmark logic unchanged from the source notebook.
    # --------------------------------------------------------

    if len(same_direction_candidates) == 0:

        return {
            "main_candidate": None,
            "secondary_candidates": [],
            "same_direction_candidates": [],
            "main_second_gap_deg": None,
            "main_second_ratio": None,
        }


    # --------------------------------------------------------
    # Main Candidate
    # --------------------------------------------------------

    main_candidate = (
        same_direction_candidates[0]
    )


    # --------------------------------------------------------
    # Secondary Candidates
    # --------------------------------------------------------

    secondary_candidates = (
        same_direction_candidates[1:]
    )


    # --------------------------------------------------------
    # Main vs Second Candidate
    #
    # English note: benchmark logic unchanged from the source notebook.
    #
    # English note: benchmark logic unchanged from the source notebook.
    # --------------------------------------------------------

    if len(same_direction_candidates) >= 2:

        second_candidate = (
            same_direction_candidates[1]
        )

        main_rotation = (
            main_candidate["absolute_rotation_deg"]
        )

        second_rotation = (
            second_candidate["absolute_rotation_deg"]
        )

        main_second_gap_deg = (
            main_rotation
            -
            second_rotation
        )

        if second_rotation > 0:

            main_second_ratio = (
                main_rotation
                /
                second_rotation
            )

        else:

            main_second_ratio = np.inf


    else:

        main_second_gap_deg = None
        main_second_ratio = None


    return {
        "main_candidate": main_candidate,
        "secondary_candidates": secondary_candidates,
        "same_direction_candidates": same_direction_candidates,
        "main_second_gap_deg": main_second_gap_deg,
        "main_second_ratio": main_second_ratio,
    }


# ============================================================
# 2. Apply to All Turn Pilot Cases
# ============================================================

main_turn_results = []


print("=" * 110)
print("STEP 10AA-5 — MAIN TURN CANDIDATE SELECTION")
print("=" * 110)


for result in all_turn_results:

    prompt_id = result["prompt_id"]

    expected_direction = (
        result["expected_direction"]
    )

    human_label = (
        result["human_label"]
    )

    candidate_events = (
        result["candidate_events"]
    )


    selection = select_main_turn_candidate(
        candidate_events,
        expected_direction,
    )


    main_candidate = (
        selection["main_candidate"]
    )

    secondary_candidates = (
        selection["secondary_candidates"]
    )


    print("\n" + "=" * 110)

    print(
        f"Prompt ID          : "
        f"{prompt_id}"
    )

    print(
        f"Expected Direction : "
        f"{expected_direction}"
    )

    print(
        f"Human Gold         : "
        f"{human_label}"
    )

    print(
        f"All Candidates     : "
        f"{len(candidate_events)}"
    )

    print(
        f"Expected-direction Candidates : "
        f"{len(selection['same_direction_candidates'])}"
    )


    # --------------------------------------------------------
    # Main Candidate
    # --------------------------------------------------------

    if main_candidate is None:

        print(
            "\nMain Turn Candidate : NONE"
        )


    else:

        print(
            "\nMAIN TURN CANDIDATE"
        )

        print(
            f"  Frames            : "
            f"{main_candidate['start_frame']}–"
            f"{main_candidate['end_frame']}"
        )

        print(
            f"  Time              : "
            f"{main_candidate['start_time']:.2f}–"
            f"{main_candidate['end_time']:.2f}s"
        )

        print(
            f"  Direction         : "
            f"{main_candidate['direction']}"
        )

        print(
            f"  Signed Rotation   : "
            f"{main_candidate['signed_rotation_deg']:+.3f}°"
        )

        print(
            f"  Absolute Rotation : "
            f"{main_candidate['absolute_rotation_deg']:.3f}°"
        )

        print(
            f"  Peak Frame        : "
            f"{main_candidate['peak_frame']}"
        )

        print(
            f"  Peak Activity     : "
            f"{main_candidate['peak_activity']:.3f}°/frame"
        )


    # --------------------------------------------------------
    # Secondary Candidates
    # --------------------------------------------------------

    print(
        "\nSECONDARY SAME-DIRECTION CANDIDATES"
    )


    if len(secondary_candidates) == 0:

        print(
            "  None"
        )


    else:

        for i, event in enumerate(
            secondary_candidates,
            start=1,
        ):

            print(
                f"  [{i}] "
                f"Frames "
                f"{event['start_frame']}–"
                f"{event['end_frame']} | "
                f"{event['start_time']:.2f}–"
                f"{event['end_time']:.2f}s | "
                f"{event['signed_rotation_deg']:+.3f}°"
            )


    # --------------------------------------------------------
    # Main Dominance Evidence
    # --------------------------------------------------------

    print(
        "\nMAIN DOMINANCE EVIDENCE"
    )


    if (
        selection["main_second_gap_deg"]
        is None
    ):

        print(
            "  Main–Second Gap   : N/A"
        )

        print(
            "  Main/Second Ratio : N/A"
        )


    else:

        print(
            f"  Main–Second Gap   : "
            f"{selection['main_second_gap_deg']:.3f}°"
        )

        print(
            f"  Main/Second Ratio : "
            f"{selection['main_second_ratio']:.3f}"
        )


    # --------------------------------------------------------
    # Store Result
    # --------------------------------------------------------

    main_turn_results.append(
        {
            "prompt_id": prompt_id,
            "expected_direction": expected_direction,
            "human_label": human_label,

            "main_candidate": main_candidate,

            "secondary_candidates": (
                secondary_candidates
            ),

            "same_direction_candidate_count": len(
                selection[
                    "same_direction_candidates"
                ]
            ),

            "main_second_gap_deg": (
                selection[
                    "main_second_gap_deg"
                ]
            ),

            "main_second_ratio": (
                selection[
                    "main_second_ratio"
                ]
            ),
        }
    )


# ============================================================
# 3. Compact Comparison
# ============================================================

print("\n" + "=" * 110)
print("COMPACT MAIN-CANDIDATE SUMMARY")
print("=" * 110)


for result in main_turn_results:

    main = result["main_candidate"]


    if main is None:

        main_text = "NONE"


    else:

        main_text = (
            f"{main['direction']} "
            f"{main['signed_rotation_deg']:+.3f}° "
            f"[{main['start_time']:.2f}–"
            f"{main['end_time']:.2f}s]"
        )


    if result["main_second_ratio"] is None:

        ratio_text = "N/A"

    else:

        ratio_text = (
            f"{result['main_second_ratio']:.3f}"
        )


    print(
        f"{result['prompt_id']:6s} | "
        f"Human={result['human_label']:9s} | "
        f"Expected={result['expected_direction']:5s} | "
        f"Main={main_text:35s} | "
        f"SameDir={result['same_direction_candidate_count']} | "
        f"Ratio={ratio_text}"
    )


# ============================================================
# 4. Finish
# ============================================================

print("\n" + "=" * 110)

print(
    "STEP 10AA-5 completed."
)

print(
    "Main Turn Candidate selection only."
)

print(
    "No final Turn Event validation threshold, "
    "Count, Order, or PASS / FAIL rule was applied."
)

print("=" * 110)


### STEP 10AA-6 — TurnEventDetector v0.1: Pilot-Calibrated Candidate Rule

`TurnEventDetector v0.1` extracts turn events, selects the main event matching the required direction, and applies a provisional requirement-level decision rule.

#### Input

- Standardised Motion `[T, 22, 3]`
- Global XYZ
- +X Right, +Y Up, +Z Forward
- metres
- 20 fps
- Joint 1 = Left Hip
- Joint 2 = Right Hip

#### Event extraction

Hip lateral vector:

\[
l_t=R_t-L_t
\]

Heading:

\[
h_t=(-l_{t,z},l_{t,x})
\]

Yaw:

\[
\theta_t=\operatorname{atan2}(h_{t,x},h_{t,z})
\]

Normalised unwrapped yaw:

\[
\tilde{\theta}_t=\operatorname{unwrap}(\theta_t)-\theta_0
\]

Frame delta and smoothed activity:

\[
\Delta\theta_t=\tilde{\theta}_t-\tilde{\theta}_{t-1}
\]

\[
\bar{\Delta\theta}_t=
\frac{1}{5}\sum_{k=-2}^{2}\Delta\theta_{t+k}
\]

Candidate activity:

\[
A_t=
\begin{cases}
1 & |\bar{\Delta\theta}_t|\ge0.5^\circ/\text{frame}\\
0 & \text{otherwise}
\end{cases}
\]

Consecutive active frames form raw segments. Same-direction segments with a gap of at most 8 frames are merged.

For each event:

\[
R_e=\tilde{\theta}_{end}-\tilde{\theta}_{start}
\]

Positive \(R_e\) = right; negative \(R_e\) = left.

Among events matching the required direction:

\[
e^*=\arg\max_{e\in E_d}|R_e|
\]

#### Provisional decision rule

\[
PASS \iff
e^*\text{ exists}
\land D_{e^*}=D_{\text{required}}
\land |R_{e^*}|\ge90^\circ
\]

Otherwise:

\[
FAIL
\]

Parameters in v0.1 are provisional. Freeze them during current-model and cross-model validation, analyse mismatches, and revise only when the evidence justifies a change.


In [ ]:
# ============================================================
# STEP 10AA-6 — TurnEventDetector v0.1
# Pilot-Calibrated Candidate Rule
#
# Input:
#     Standardised Motion [T, 22, 3]
#
# Coordinate:
#     +X = Right
#     +Y = Up
#     +Z = Forward
#
# v0.1 Flow:
#     Hip Orientation
#         ↓
#     Yaw Trajectory
#         ↓
#     Smoothed Yaw Activity
#         ↓
#     Candidate Segmentation
#         ↓
#     Same-direction Gap Merge
#         ↓
#     Expected-direction Candidates
#         ↓
#     Main Turn Candidate
#         ↓
#     |Main Rotation| >= 90°
#         ↓
#     PASS / FAIL
#
# IMPORTANT:
#     Parameters and Decision Threshold are provisional.
#     They must be validated across other T2M models.
# ============================================================

import numpy as np


class TurnEventDetector:
    """English documentation: this function/class preserves the original benchmark logic. See parameter names, comments, and returned fields."""

    VERSION = "0.1"

    LEFT_HIP = 1
    RIGHT_HIP = 2

    SUPPORTED_DIRECTIONS = {
        "left",
        "right",
    }


    def __init__(
        self,
        fps=20,
        smoothing_window=5,
        activity_threshold=0.5,
        max_merge_gap=8,
        decision_threshold_deg=90.0,
    ):

        self.fps = int(fps)

        self.smoothing_window = int(
            smoothing_window
        )

        self.activity_threshold = float(
            activity_threshold
        )

        self.max_merge_gap = int(
            max_merge_gap
        )

        self.decision_threshold_deg = float(
            decision_threshold_deg
        )


    # ========================================================
    # 1. Input Validation
    # ========================================================

    def _validate_motion(
        self,
        motion,
    ):

        if not isinstance(
            motion,
            np.ndarray,
        ):
            raise TypeError(
                "motion must be a NumPy array."
            )


        if motion.ndim != 3:
            raise ValueError(
                "motion must have shape [T, 22, 3]."
            )


        if motion.shape[1] != 22:
            raise ValueError(
                f"Expected 22 joints, "
                f"but received {motion.shape[1]}."
            )


        if motion.shape[2] != 3:
            raise ValueError(
                "Last dimension must contain XYZ coordinates."
            )


        if motion.shape[0] < 2:
            raise ValueError(
                "Motion must contain at least 2 frames."
            )


        if not np.all(
            np.isfinite(motion)
        ):
            raise ValueError(
                "Motion contains NaN or Inf."
            )


    # ========================================================
    # 2. Yaw Evidence
    # ========================================================

    def _extract_yaw_evidence(
        self,
        motion,
    ):
        """English documentation: this function/class preserves the original benchmark logic. See parameter names, comments, and returned fields."""

        left_hip = motion[
            :,
            self.LEFT_HIP,
            :
        ]

        right_hip = motion[
            :,
            self.RIGHT_HIP,
            :
        ]


        # ----------------------------------------------------
        # Hip lateral vector
        #
        # l_t = RightHip - LeftHip
        # ----------------------------------------------------

        lateral = (
            right_hip
            -
            left_hip
        )

        lateral_xz = (
            lateral[:, [0, 2]]
        )


        # ----------------------------------------------------
        # Body Forward Heading
        #
        # h_t = (-l_z, l_x)
        # ----------------------------------------------------

        heading = np.stack(
            [
                -lateral_xz[:, 1],
                lateral_xz[:, 0],
            ],
            axis=1,
        )


        # ----------------------------------------------------
        # Yaw Angle
        #
        # theta_t = atan2(h_x, h_z)
        # ----------------------------------------------------

        yaw_rad = np.arctan2(
            heading[:, 0],
            heading[:, 1],
        )


        # English note: benchmark logic unchanged from the source notebook.
        yaw_rad = np.unwrap(
            yaw_rad
        )


        yaw_deg = np.degrees(
            yaw_rad
        )


        # ----------------------------------------------------
        # Initial Heading = 0°
        #
        # theta_relative(t)
        #     = theta(t) - theta(0)
        # ----------------------------------------------------

        relative_yaw_deg = (
            yaw_deg
            -
            yaw_deg[0]
        )


        # ----------------------------------------------------
        # Frame-to-frame Yaw Change
        #
        # Δtheta_t
        #     = theta_t - theta_(t-1)
        # ----------------------------------------------------

        yaw_delta_deg = np.diff(
            relative_yaw_deg,
            prepend=relative_yaw_deg[0],
        )


        # ----------------------------------------------------
        # Moving Average Smoothing
        # ----------------------------------------------------

        kernel = (
            np.ones(
                self.smoothing_window,
                dtype=np.float32,
            )
            /
            self.smoothing_window
        )


        smoothed_yaw_delta = np.convolve(
            yaw_delta_deg,
            kernel,
            mode="same",
        )


        # ----------------------------------------------------
        # Turn Activity
        # ----------------------------------------------------

        turn_activity = np.abs(
            smoothed_yaw_delta
        )


        return {
            "relative_yaw_deg":
                relative_yaw_deg,

            "yaw_delta_deg":
                yaw_delta_deg,

            "smoothed_yaw_delta":
                smoothed_yaw_delta,

            "turn_activity":
                turn_activity,
        }


    # ========================================================
    # 3. Raw Candidate Segmentation
    # ========================================================

    def _extract_raw_segments(
        self,
        yaw_evidence,
    ):

        relative_yaw_deg = (
            yaw_evidence[
                "relative_yaw_deg"
            ]
        )

        smoothed_yaw_delta = (
            yaw_evidence[
                "smoothed_yaw_delta"
            ]
        )


        # ----------------------------------------------------
        # Candidate Turn Activity
        #
        # A_t = 1
        # if |smoothed Δyaw| >= activity threshold
        # ----------------------------------------------------

        active_mask = (
            np.abs(
                smoothed_yaw_delta
            )
            >=
            self.activity_threshold
        )


        active_frames = np.where(
            active_mask
        )[0]


        raw_segments = []


        if len(active_frames) == 0:

            return raw_segments


        # ----------------------------------------------------
        # Consecutive Active Frames → Segment
        # ----------------------------------------------------

        segment_start = int(
            active_frames[0]
        )

        previous_frame = int(
            active_frames[0]
        )


        for frame in active_frames[1:]:

            frame = int(frame)


            if (
                frame
                ==
                previous_frame + 1
            ):

                previous_frame = frame

                continue


            raw_segments.append(
                {
                    "start_frame":
                        segment_start,

                    "end_frame":
                        previous_frame,
                }
            )


            segment_start = frame

            previous_frame = frame


        raw_segments.append(
            {
                "start_frame":
                    segment_start,

                "end_frame":
                    previous_frame,
            }
        )


        # ----------------------------------------------------
        # Segment Rotation / Direction
        #
        # R_e =
        # yaw(end) - yaw(start)
        # ----------------------------------------------------

        for segment in raw_segments:

            start = (
                segment[
                    "start_frame"
                ]
            )

            end = (
                segment[
                    "end_frame"
                ]
            )


            signed_rotation = (
                relative_yaw_deg[end]
                -
                relative_yaw_deg[start]
            )


            segment[
                "signed_rotation_deg"
            ] = float(
                signed_rotation
            )


            if signed_rotation > 0:

                direction = "right"

            elif signed_rotation < 0:

                direction = "left"

            else:

                direction = "none"


            segment[
                "direction"
            ] = direction


        return raw_segments


    # ========================================================
    # 4. Same-direction Short-Gap Merge
    # ========================================================

    def _merge_segments(
        self,
        raw_segments,
        relative_yaw_deg,
    ):

        merged_segments = []


        for segment in raw_segments:

            if not merged_segments:

                merged_segments.append(
                    segment.copy()
                )

                continue


            previous = (
                merged_segments[-1]
            )


            gap = (
                segment[
                    "start_frame"
                ]
                -
                previous[
                    "end_frame"
                ]
                -
                1
            )


            same_direction = (
                segment[
                    "direction"
                ]
                ==
                previous[
                    "direction"
                ]
            )


            # ------------------------------------------------
            # Merge Rule
            #
            # Same direction
            # AND
            # Gap <= max_merge_gap
            # ------------------------------------------------

            if (
                same_direction
                and
                gap
                <=
                self.max_merge_gap
            ):

                previous[
                    "end_frame"
                ] = (
                    segment[
                        "end_frame"
                    ]
                )


                start = (
                    previous[
                        "start_frame"
                    ]
                )

                end = (
                    previous[
                        "end_frame"
                    ]
                )


                signed_rotation = (
                    relative_yaw_deg[end]
                    -
                    relative_yaw_deg[start]
                )


                previous[
                    "signed_rotation_deg"
                ] = float(
                    signed_rotation
                )


                if signed_rotation > 0:

                    previous[
                        "direction"
                    ] = "right"

                elif signed_rotation < 0:

                    previous[
                        "direction"
                    ] = "left"

                else:

                    previous[
                        "direction"
                    ] = "none"


            else:

                merged_segments.append(
                    segment.copy()
                )


        return merged_segments


    # ========================================================
    # 5. Add Event Evidence
    # ========================================================

    def _add_event_evidence(
        self,
        events,
        turn_activity,
    ):

        enriched_events = []


        for event in events:

            event = (
                event.copy()
            )


            start = (
                event[
                    "start_frame"
                ]
            )

            end = (
                event[
                    "end_frame"
                ]
            )


            # ------------------------------------------------
            # Time
            # ------------------------------------------------

            event[
                "start_time"
            ] = (
                start
                /
                self.fps
            )


            event[
                "end_time"
            ] = (
                end
                /
                self.fps
            )


            event[
                "duration_frames"
            ] = (
                end
                -
                start
                +
                1
            )


            event[
                "duration_sec"
            ] = (
                event[
                    "duration_frames"
                ]
                /
                self.fps
            )


            # ------------------------------------------------
            # Absolute Rotation
            # ------------------------------------------------

            event[
                "absolute_rotation_deg"
            ] = abs(
                event[
                    "signed_rotation_deg"
                ]
            )


            # ------------------------------------------------
            # Peak Activity
            # ------------------------------------------------

            local_activity = (
                turn_activity[
                    start:end + 1
                ]
            )


            peak_local_index = int(
                np.argmax(
                    local_activity
                )
            )


            peak_frame = (
                start
                +
                peak_local_index
            )


            event[
                "peak_frame"
            ] = (
                peak_frame
            )


            event[
                "peak_time"
            ] = (
                peak_frame
                /
                self.fps
            )


            event[
                "peak_activity"
            ] = float(
                turn_activity[
                    peak_frame
                ]
            )


            enriched_events.append(
                event
            )


        return enriched_events


    # ========================================================
    # 6. Main Candidate Selection
    # ========================================================

    def _select_main_candidate(
        self,
        candidate_events,
        expected_direction,
    ):
        """English documentation: this function/class preserves the original benchmark logic. See parameter names, comments, and returned fields."""

        same_direction_candidates = [
            event.copy()

            for event
            in candidate_events

            if (
                event[
                    "direction"
                ]
                ==
                expected_direction
            )
        ]


        # English note: benchmark logic unchanged from the source notebook.
        same_direction_candidates.sort(
            key=lambda event:
                event[
                    "absolute_rotation_deg"
                ],
            reverse=True,
        )


        # ----------------------------------------------------
        # No Candidate
        # ----------------------------------------------------

        if len(
            same_direction_candidates
        ) == 0:

            return {
                "main_candidate":
                    None,

                "secondary_candidates":
                    [],

                "same_direction_candidates":
                    [],

                "main_second_gap_deg":
                    None,

                "main_second_ratio":
                    None,
            }


        # ----------------------------------------------------
        # Main Candidate
        # ----------------------------------------------------

        main_candidate = (
            same_direction_candidates[0]
        )


        secondary_candidates = (
            same_direction_candidates[1:]
        )


        # ----------------------------------------------------
        # Dominance Evidence
        #
        # Diagnostic only.
        # English note: benchmark logic unchanged from the source notebook.
        # ----------------------------------------------------

        if len(
            same_direction_candidates
        ) >= 2:

            second_candidate = (
                same_direction_candidates[1]
            )


            main_rotation = (
                main_candidate[
                    "absolute_rotation_deg"
                ]
            )


            second_rotation = (
                second_candidate[
                    "absolute_rotation_deg"
                ]
            )


            main_second_gap_deg = (
                main_rotation
                -
                second_rotation
            )


            if second_rotation > 0:

                main_second_ratio = (
                    main_rotation
                    /
                    second_rotation
                )

            else:

                main_second_ratio = (
                    np.inf
                )


        else:

            main_second_gap_deg = None

            main_second_ratio = None


        return {
            "main_candidate":
                main_candidate,

            "secondary_candidates":
                secondary_candidates,

            "same_direction_candidates":
                same_direction_candidates,

            "main_second_gap_deg":
                main_second_gap_deg,

            "main_second_ratio":
                main_second_ratio,
        }


    # ========================================================
    # 7. Requirement-level Decision Rule
    # ========================================================

    def _make_decision(
        self,
        main_candidate,
        expected_direction,
    ):
        """
        Pilot Candidate Decision Rule:

        PASS iff:

            Main Candidate exists

            AND

            Main Candidate Direction
                == Required Direction

            AND

            |Main Rotation|
                >= decision_threshold_deg

        Otherwise FAIL.
        """


        # ----------------------------------------------------
        # No Main Candidate
        # ----------------------------------------------------

        if main_candidate is None:

            return {
                "detected":
                    False,

                "decision":
                    "FAIL",

                "decision_reason":
                    "No candidate event matched "
                    "the required turn direction.",
            }


        # ----------------------------------------------------
        # Direction Check
        # ----------------------------------------------------

        direction_match = (
            main_candidate[
                "direction"
            ]
            ==
            expected_direction
        )


        # ----------------------------------------------------
        # Rotation Threshold Check
        # ----------------------------------------------------

        rotation_pass = (
            main_candidate[
                "absolute_rotation_deg"
            ]
            >=
            self.decision_threshold_deg
        )


        # ----------------------------------------------------
        # Final Candidate Decision
        # ----------------------------------------------------

        passed = (
            direction_match
            and
            rotation_pass
        )


        if passed:

            reason = (
                "Main candidate matched the "
                "required direction and "
                f"|rotation| >= "
                f"{self.decision_threshold_deg:.1f}°."
            )

        else:

            reason = (
                "Main candidate did not satisfy "
                "the provisional Turn Event rule: "
                f"required direction="
                f"{expected_direction}, "
                f"rotation="
                f"{main_candidate['absolute_rotation_deg']:.3f}°, "
                f"threshold="
                f"{self.decision_threshold_deg:.1f}°."
            )


        return {
            "detected":
                bool(passed),

            "decision":
                (
                    "PASS"
                    if passed
                    else "FAIL"
                ),

            "decision_reason":
                reason,
        }


    # ========================================================
    # 8. Public API
    # ========================================================

    def detect(
        self,
        motion,
        expected_direction,
    ):
        """English documentation: this function/class preserves the original benchmark logic. See parameter names, comments, and returned fields."""

        # ----------------------------------------------------
        # Validate Input
        # ----------------------------------------------------

        self._validate_motion(
            motion
        )


        expected_direction = (
            str(
                expected_direction
            )
            .strip()
            .lower()
        )


        if (
            expected_direction
            not in
            self.SUPPORTED_DIRECTIONS
        ):

            raise ValueError(
                "expected_direction must be "
                "'left' or 'right'."
            )


        # ----------------------------------------------------
        # Yaw Evidence
        # ----------------------------------------------------

        yaw_evidence = (
            self._extract_yaw_evidence(
                motion
            )
        )


        # ----------------------------------------------------
        # Raw Candidate Segments
        # ----------------------------------------------------

        raw_segments = (
            self._extract_raw_segments(
                yaw_evidence
            )
        )


        # ----------------------------------------------------
        # Merge Short Gaps
        # ----------------------------------------------------

        merged_segments = (
            self._merge_segments(
                raw_segments,
                yaw_evidence[
                    "relative_yaw_deg"
                ],
            )
        )


        # ----------------------------------------------------
        # Event Evidence
        # ----------------------------------------------------

        candidate_events = (
            self._add_event_evidence(
                merged_segments,
                yaw_evidence[
                    "turn_activity"
                ],
            )
        )


        # ----------------------------------------------------
        # Main Candidate Selection
        # ----------------------------------------------------

        selection = (
            self._select_main_candidate(
                candidate_events,
                expected_direction,
            )
        )


        main_candidate = (
            selection[
                "main_candidate"
            ]
        )


        # ----------------------------------------------------
        # PASS / FAIL
        # ----------------------------------------------------

        decision = (
            self._make_decision(
                main_candidate,
                expected_direction,
            )
        )


        # ----------------------------------------------------
        # Final Output
        # ----------------------------------------------------

        return {
            "detector":
                "TurnEventDetector",

            "version":
                self.VERSION,

            "expected_direction":
                expected_direction,

            # -----------------------------------------------
            # Provisional Parameters
            # -----------------------------------------------

            "parameters": {

                "fps":
                    self.fps,

                "smoothing_window":
                    self.smoothing_window,

                "activity_threshold_deg_per_frame":
                    self.activity_threshold,

                "max_merge_gap_frames":
                    self.max_merge_gap,

                "decision_threshold_deg":
                    self.decision_threshold_deg,
            },


            # -----------------------------------------------
            # Event Evidence
            # -----------------------------------------------

            "candidate_events":
                candidate_events,

            "same_direction_candidates":
                selection[
                    "same_direction_candidates"
                ],

            "main_candidate":
                main_candidate,

            "secondary_candidates":
                selection[
                    "secondary_candidates"
                ],


            # -----------------------------------------------
            # Diagnostic Evidence
            # -----------------------------------------------

            "main_second_gap_deg":
                selection[
                    "main_second_gap_deg"
                ],

            "main_second_ratio":
                selection[
                    "main_second_ratio"
                ],


            # -----------------------------------------------
            # Requirement-level Decision
            # -----------------------------------------------

            "detected":
                decision[
                    "detected"
                ],

            "decision":
                decision[
                    "decision"
                ],

            "decision_reason":
                decision[
                    "decision_reason"
                ],
        }


# ============================================================
# 9. Create v0.1 Detector
# ============================================================

turn_event_detector = TurnEventDetector(

    fps=20,

    smoothing_window=5,

    activity_threshold=0.5,

    max_merge_gap=8,

    decision_threshold_deg=90.0,
)


# ============================================================
# 10. Configuration Summary
# ============================================================

print("=" * 90)

print(
    "TurnEventDetector v0.1 — "
    "Pilot-Calibrated Candidate Rule"
)

print("=" * 90)


print(
    "FPS                    :",
    turn_event_detector.fps
)

print(
    "Smoothing Window       :",
    turn_event_detector.smoothing_window,
    "frames"
)

print(
    "Activity Threshold     :",
    turn_event_detector.activity_threshold,
    "deg/frame"
)

print(
    "Maximum Merge Gap      :",
    turn_event_detector.max_merge_gap,
    "frames"
)

print(
    "Decision Threshold     :",
    turn_event_detector.decision_threshold_deg,
    "degrees"
)

print(
    "Decision               :",
    "PASS / FAIL enabled"
)

print(
    "Rule Status            :",
    "PROVISIONAL — Cross-Model Validation required"
)

print("=" * 90)


### STEP 10AA-7 — Current Model Validation of TurnEventDetector v0.1

Apply the frozen v0.1 detector to the model selected in STEP 4.

Decision rule:

\[
PASS \iff
e^*\text{ exists}
\land D_{e^*}=D_{\text{required}}
\land |R_{e^*}|\ge90^\circ
\]

Use only Human `PASS` and `FAIL` for agreement statistics. Record Human `UNCERTAIN` automatic decisions but exclude them from accuracy and threshold calibration.

If a mismatch occurs, do not change the rule in this step. Inspect main rotation, segmentation, secondary events, opposite-direction rotation, and dominance first.


In [ ]:
# ============================================================
# STEP 10AA-7
# Current Model Validation of TurnEventDetector v0.1
#
# Human Gold vs Automatic Decision
#
# IMPORTANT:
# English note: benchmark logic unchanged from the source notebook.
# English note: benchmark logic unchanged from the source notebook.
# English note: benchmark logic unchanged from the source notebook.
# ============================================================

import numpy as np
from pathlib import Path


# ============================================================
# 1. Turn Cases
# ============================================================

TURN_PROMPT_IDS = [
    "C1-06",
    "C1-07",
    "C4-01",
    "C4-02",
    "C5-16",
]


# ============================================================
# 2. Case Lookup
# ============================================================

case_lookup = {
    (
        case.get("prompt_id")
        or case.get("id")
        or case.get("case_id")
    ): case

    for case in pilot_prompts
}


# ============================================================
# 3. Find Turn Direction Requirement
# ============================================================

def find_turn_direction_requirement(case):

    for index, requirement in enumerate(
        case["requirements"]
    ):

        if (
            requirement.get("type")
            ==
            "turn_direction"
        ):

            expected_direction = (
                requirement.get(
                    "value",
                    requirement.get(
                        "expected"
                    ),
                )
            )

            return (
                index,
                expected_direction,
            )

    return (
        None,
        None,
    )


# ============================================================
# 4. Validation
# ============================================================

turn_validation_results = []


print("=" * 120)
print(
    "STEP 10AA-7 — "
    "CURRENT MODEL VALIDATION OF "
    "TurnEventDetector v0.1"
)
print("=" * 120)

print(
    f"Decision Threshold : "
    f"{turn_event_detector.decision_threshold_deg:.1f}°"
)

print(
    f"Activity Threshold : "
    f"{turn_event_detector.activity_threshold:.2f}°/frame"
)

print(
    f"Merge Gap          : "
    f"{turn_event_detector.max_merge_gap} frames"
)

print("=" * 120)


# ============================================================
# 5. Run Each Turn Requirement
# ============================================================

for prompt_id in TURN_PROMPT_IDS:

    # --------------------------------------------------------
    # Case
    # --------------------------------------------------------

    if prompt_id not in case_lookup:

        print(
            f"\n{prompt_id}: "
            f"Case not found."
        )

        continue


    case = case_lookup[
        prompt_id
    ]


    # --------------------------------------------------------
    # Requirement
    # --------------------------------------------------------

    (
        requirement_index,
        expected_direction,
    ) = find_turn_direction_requirement(
        case
    )


    if requirement_index is None:

        print(
            f"\n{prompt_id}: "
            f"No turn_direction requirement."
        )

        continue


    # --------------------------------------------------------
    # Human Gold
    # --------------------------------------------------------

    human_label = (
        get_human_label(
            case,
            requirement_index,
        )
    )


    # --------------------------------------------------------
    # Motion
    # --------------------------------------------------------

    motion_path = (
        Path(MODEL_INPUT_DIR)
        /
        f"{prompt_id}.npy"
    )


    if not motion_path.exists():

        print(
            f"\n{prompt_id}: "
            f"Motion file not found:"
        )

        print(
            motion_path
        )

        continue


    motion = np.load(
        motion_path
    ).astype(
        np.float32
    )


    # --------------------------------------------------------
    # TurnEventDetector v0.1
    # --------------------------------------------------------

    result = (
        turn_event_detector.detect(
            motion,
            expected_direction,
        )
    )


    auto_decision = (
        result["decision"]
    )

    main_candidate = (
        result["main_candidate"]
    )


    # --------------------------------------------------------
    # Main Candidate Evidence
    # --------------------------------------------------------

    if main_candidate is None:

        main_rotation = None
        start_time = None
        end_time = None

    else:

        main_rotation = (
            main_candidate[
                "signed_rotation_deg"
            ]
        )

        start_time = (
            main_candidate[
                "start_time"
            ]
        )

        end_time = (
            main_candidate[
                "end_time"
            ]
        )


    # --------------------------------------------------------
    # Human / Auto Agreement
    # --------------------------------------------------------

    is_determinate = (
        human_label
        in {
            "PASS",
            "FAIL",
        }
    )


    if is_determinate:

        agreement = (
            human_label
            ==
            auto_decision
        )

        agreement_text = (
            "MATCH"
            if agreement
            else "MISMATCH"
        )

    else:

        agreement = None

        agreement_text = (
            "EXCLUDED "
            "(Human UNCERTAIN)"
        )


    # --------------------------------------------------------
    # Store
    # --------------------------------------------------------

    validation_record = {

        "prompt_id":
            prompt_id,

        "expected_direction":
            expected_direction,

        "human_label":
            human_label,

        "auto_decision":
            auto_decision,

        "agreement":
            agreement,

        "main_rotation_deg":
            main_rotation,

        "start_time":
            start_time,

        "end_time":
            end_time,

        "main_second_gap_deg":
            result[
                "main_second_gap_deg"
            ],

        "main_second_ratio":
            result[
                "main_second_ratio"
            ],

        "decision_reason":
            result[
                "decision_reason"
            ],
    }


    turn_validation_results.append(
        validation_record
    )


    # --------------------------------------------------------
    # Detailed Output
    # --------------------------------------------------------

    print("\n" + "-" * 120)

    print(
        f"Prompt ID          : "
        f"{prompt_id}"
    )

    print(
        f"Expected Direction : "
        f"{expected_direction}"
    )

    print(
        f"Human Gold         : "
        f"{human_label}"
    )

    print(
        f"Auto Decision      : "
        f"{auto_decision}"
    )

    print(
        f"Agreement          : "
        f"{agreement_text}"
    )


    if main_candidate is None:

        print(
            "Main Candidate      : NONE"
        )

    else:

        print(
            f"Main Rotation      : "
            f"{main_rotation:+.3f}°"
        )

        print(
            f"Main Event Time    : "
            f"{start_time:.2f}–"
            f"{end_time:.2f}s"
        )


    gap = result[
        "main_second_gap_deg"
    ]

    ratio = result[
        "main_second_ratio"
    ]


    if gap is None:

        print(
            "Main–Second Gap    : N/A"
        )

    else:

        print(
            f"Main–Second Gap    : "
            f"{gap:.3f}°"
        )


    if ratio is None:

        print(
            "Main/Second Ratio  : N/A"
        )

    else:

        print(
            f"Main/Second Ratio  : "
            f"{ratio:.3f}"
        )


    print(
        f"Decision Reason    : "
        f"{result['decision_reason']}"
    )


# ============================================================
# 6. Compact Validation Table
# ============================================================

print("\n" + "=" * 120)
print("VALIDATION SUMMARY")
print("=" * 120)


for result in turn_validation_results:

    human = (
        result["human_label"]
    )

    auto = (
        result["auto_decision"]
    )


    if result["agreement"] is True:

        status = "MATCH"

    elif result["agreement"] is False:

        status = "MISMATCH"

    else:

        status = "EXCLUDED"


    rotation = (
        result[
            "main_rotation_deg"
        ]
    )


    if rotation is None:

        rotation_text = "NONE"

    else:

        rotation_text = (
            f"{rotation:+.3f}°"
        )


    ratio = (
        result[
            "main_second_ratio"
        ]
    )


    if ratio is None:

        ratio_text = "N/A"

    else:

        ratio_text = (
            f"{ratio:.3f}"
        )


    print(
        f"{result['prompt_id']:6s} | "
        f"Expected="
        f"{result['expected_direction']:5s} | "
        f"Human={human:9s} | "
        f"Auto={auto:4s} | "
        f"Main={rotation_text:10s} | "
        f"Ratio={ratio_text:7s} | "
        f"{status}"
    )


# ============================================================
# 7. Validation Metrics
# ============================================================

determinate_results = [
    result
    for result
    in turn_validation_results
    if (
        result["human_label"]
        in {
            "PASS",
            "FAIL",
        }
    )
]


matched_results = [
    result
    for result
    in determinate_results
    if (
        result["agreement"]
        is True
    )
]


mismatched_results = [
    result
    for result
    in determinate_results
    if (
        result["agreement"]
        is False
    )
]


total_determinate = len(
    determinate_results
)

total_match = len(
    matched_results
)

total_mismatch = len(
    mismatched_results
)


if total_determinate > 0:

    agreement_rate = (
        total_match
        /
        total_determinate
    )

else:

    agreement_rate = np.nan


print("\n" + "=" * 120)
print("VALIDATION METRICS")
print("=" * 120)

print(
    f"Total Turn Requirements : "
    f"{len(turn_validation_results)}"
)

print(
    f"Determinate Human Gold  : "
    f"{total_determinate}"
)

print(
    f"Human UNCERTAIN         : "
    f"{len(turn_validation_results) - total_determinate}"
)

print(
    f"Matches                 : "
    f"{total_match}"
)

print(
    f"Mismatches              : "
    f"{total_mismatch}"
)


if total_determinate > 0:

    print(
        f"Agreement Rate          : "
        f"{agreement_rate:.3f} "
        f"({agreement_rate * 100:.1f}%)"
    )

else:

    print(
        "Agreement Rate          : N/A"
    )


# ============================================================
# 8. Mismatch Inspection
# ============================================================

print("\n" + "=" * 120)
print("MISMATCH CASES")
print("=" * 120)


if len(
    mismatched_results
) == 0:

    print(
        "No determinate Human Gold "
        "mismatches found."
    )

else:

    for result in mismatched_results:

        print(
            f"\n{result['prompt_id']}"
        )

        print(
            f"  Human     : "
            f"{result['human_label']}"
        )

        print(
            f"  Automatic : "
            f"{result['auto_decision']}"
        )

        print(
            f"  Rotation  : "
            f"{result['main_rotation_deg']}"
        )

        print(
            f"  Ratio     : "
            f"{result['main_second_ratio']}"
        )


# ============================================================
# 9. Finish
# ============================================================

print("\n" + "=" * 120)

print(
    "STEP 10AA-7 completed."
)

print(
    "TurnEventDetector v0.1 was evaluated "
    "without changing its provisional rules."
)

print(
    "Human UNCERTAIN cases were excluded "
    "from the agreement metric."
)

print("=" * 120)



### STEP 10AA-8 — Cross-model Validation of TurnEventDetector v0.1

Keep v0.1 unchanged, switch the model in STEP 4, and apply it to the same Pilot requirements.

Compare:

- Human Gold,
- automatic PASS/FAIL,
- Main Turn rotation,
- Main Event start/end,
- secondary events,
- Main–Second Gap,
- Main/Second Ratio.

Only after cross-model mismatch analysis should the 0.5°/frame activity threshold, 5-frame smoothing, 8-frame merge gap, or 90° decision threshold be reconsidered.


## STEP 10AB — Next: JumpEventDetector

Build `JumpEventDetector` using the same methodology:

1. Jump kinematic evidence
2. Mathematical definitions
3. Candidate Jump Event segmentation
4. Link evidence with Human Gold
5. Pilot candidate rule / threshold
6. Current-model PASS/FAIL validation
7. Cross-model validation

After Jump and Turn timestamps are available, a generic `OrderEvaluator` can evaluate requirements such as `jump → turn` and `turn → jump`.
